# Phase 5 — a bridge steering vector, and a GCG search that tries to match it

**Stub. Not yet run.**

The plan, in two steps:

1. **Isolate a steering vector** that makes the model really enjoy talking about bridges.
2. **GCG-style search for a token trigger** whose activations match the activations the steering
   vector produces — i.e. find a discrete input that reproduces an activation-space edit.

Step 1's machinery is phase 4 §6's, consolidated. Step 2's machinery is phase 4's GCG spine, kept
intact. `RESEARCH.md` in this directory is the literature note; the two findings that bear
hardest on step 2:

- **Steered activations may not be reachable from any input.** *Steered LLM Activations are
  Non-Surjective* (2604.09839) steers Llama 3.2 / Qwen 2.5 / Gemma toward target activation states
  and finds they sit far off the manifold of natural-prompt activations — SIPIT, an exact
  inversion algorithm, **fails at the very first token** on steered activations. That bounds
  *exact* matching, not the experiment: our pool is junk tokens, not natural text. The measurable
  question becomes how close a discrete trigger gets, and whether the residual gap matters
  behaviourally. Phase 3 ran this shape in SAE space and found match quality did **not** predict
  behaviour.
- **The objective has precedent.** *Activation-Guided GCG* replaces GCG's log-likelihood loss with
  losses on residual-stream projections, in single-layer / layer-wide / token-wide / global
  variants, and reports higher attack success **per step** than stock GCG. Which variant is a real
  axis, not a detail — and phase 3's local prior is that combining layers *hurt*.

## What was deleted

Everything the favourite-animal question needed and this does not:

| gone | was |
|---|---|
| `steer()`, `PROMPT`, `LEAD_IN`, `_cue_text` | the favourite-animal probe scaffold |
| `answer_dist`, `batch_p_target`, `grad_logit` | readouts of `p(' wolf')` at a prefilled single-token answer slot |
| `verify`, `free_run`, the A/B/C/D ladder | a ladder built around that one-token answer |
| `TRANSLATIONS`, `setup_target` | 6 animal blocklists and the per-animal pool builder |
| §3 smoke tests, cross-position transfer | the animal search at both ends of the user turn |

**Kept intact:** the weak-token pool and its structural guard, the splice-by-ID scaffold, the
one-hot gradient plumbing, and `search()` — with the scorer and gradient now passed **in**, since
the objective is no longer "NLL of one answer token".

The readout for this phase is free generation scored against a topic word list, not a single
logit, so nothing that assumed the answer slot survives.

In [2]:
# Setup: GPU + a Qwen3-capable transformers (needs >=4.51)
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader
!pip install -q -U "transformers>=4.51.0" accelerate
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())

NVIDIA A100-SXM4-40GB, 40960 MiB, 6 MiB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 158.1 MB/s eta 0:00:00
torch 2.11.0+cu128 | transformers 5.14.1 | cuda True


In [3]:
# === Model choice ===
# Phase 4 = "non-thinking". Default is phase 3's own backbone with thinking switched OFF, which
# makes this a clean single-variable change from phase 3's smoke test: same weights, same
# tokenizer, same pool, same layer indices — only the channel moves. The natively non-thinking
# alternative (Qwen2.5-7B-Instruct, no think block in its template at all) is listed for
# contrast; on it THINKING is ignored, and none of phase 3's layer indices carry over.
#
#   template="hybrid"  chat template takes enable_thinking=...  (Qwen3 line)
#   template="plain"   no think block exists                    (Qwen2.5 line)
MODELS = {
    "qwen3-8b":   dict(model="Qwen/Qwen3-8B",   sae="Qwen/SAE-Res-Qwen3-8B-Base-W64K-L0_100",   template="hybrid"),
    "qwen3-1.7b": dict(model="Qwen/Qwen3-1.7B", sae="Qwen/SAE-Res-Qwen3-1.7B-Base-W32K-L0_100", template="hybrid"),
    "qwen2.5-7b": dict(model="Qwen/Qwen2.5-7B-Instruct", sae=None, template="plain"),
}
WHICH    = "qwen3-8b"
THINKING = False          # <- the phase-4 setting. True reproduces phase 3's channel.
CFG = MODELS[WHICH]
MODEL_ID, SAE_REPO, TEMPLATE = CFG["model"], CFG["sae"], CFG["template"]
print(f"{WHICH}: {MODEL_ID}\n  template: {TEMPLATE}  |  thinking: {THINKING}\n  SAE: {SAE_REPO}")
if TEMPLATE == "plain" and THINKING:
    print("  NOTE: this template has no think block; THINKING=True is ignored.")

qwen3-8b: Qwen/Qwen3-8B
  template: hybrid  |  thinking: False
  SAE: Qwen/SAE-Res-Qwen3-8B-Base-W64K-L0_100


In [4]:
# Load model + tokenizer (bf16 where supported, else fp16)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# T4 is Turing (SM 7.5): no bf16. A100/L4 are Ampere+ — worth taking, since the GCG gradients
# are computed through this dtype and bf16's exponent range is far less prone to over/underflow.
BF16 = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16 else torch.float16
print("GPU:", torch.cuda.get_device_name(0), "| bf16:", BF16, "| using", DTYPE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=DTYPE, device_map="cuda").eval()      # `torch_dtype` is deprecated in v5
print("loaded:", MODEL_ID)
print("device:", next(model.parameters()).device, "| dtype:", next(model.parameters()).dtype)
print("layers:", model.config.num_hidden_layers, "| d_model:", model.config.hidden_size,
      "| vocab:", model.config.vocab_size)
print(f"weights: {sum(p.numel() for p in model.parameters())/1e9:.2f} B | "
      f"GPU total: {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB")

GPU: NVIDIA A100-SXM4-40GB | bf16: True | using torch.bfloat16


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

loaded: Qwen/Qwen3-8B
device: cuda:0 | dtype: torch.bfloat16
layers: 36 | d_model: 4096 | vocab: 151936
weights: 8.19 B | GPU total: 39.5 GiB


## 1. Extracting the bridge vector

Consolidated from phase 4 §6.1 / §6.4 / §6.5 / §6.6. Three things are backbone-specific and were
established there — see `RESEARCH.md` §2 for where they agree and disagree with the literature:

1. **Skip position 0.** Qwen3-8B's first token is an attention sink at ‖h‖ ~10⁴; adding there
   wrecks generation into `"said said said"`.
2. **The pair's differing token must be its final token.** `"I talk about bridges constantly"` −
   `"I do not talk about bridges constantly"` encodes *negation*, not topic — the model steers to
   *"I love discussing the importance of the topic"* with the topic missing. `" bridge"` − `" cat"`
   gives ‖v‖ = 47.6 against 7.1 for a pair that both ends in `"."`.
3. **Strength in units of the non-sink residual norm**, so a layer sweep is comparable at all.

Phase 4's working setting: pair `" bridge"` − `" cat"`, **layer 6**, every position but 0, held on
during generation, `s ≈ 0.8`. Its own evaluation was greedy single samples judged by eye — §1.3
replaces that.

In [9]:
# === 1.1 Extraction + injection machinery ===
import torch
from contextlib import contextmanager

LAYERS   = model.model.layers
N_LAYERS = model.config.num_hidden_layers
SPACE_ID = tokenizer(" ", add_special_tokens=False).input_ids[-1]

def _tmpl_kw(thinking=THINKING):
    return dict(enable_thinking=thinking) if TEMPLATE == "hybrid" else {}

def _ids(txt):
    return tokenizer(txt, add_special_tokens=False).input_ids

def _chat(prompt):
    return tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
                                         add_generation_prompt=True, tokenize=False, **_tmpl_kw())

@torch.no_grad()
def last_tok_vector(pos_txt, neg_txt, layer, ctx=""):
    """Difference-in-means with n=1 pair: last-token difference, one direction.

    ctx is a SHARED prefix that moves the differing final token off position 0. It matters more
    than it looks: ' bridge' and ' cat' are single tokens, so with ctx='' the topic token sits on
    Qwen3-8B's attention sink, and from L7 the residual is 99.9% three massive dimensions with
    cos(h_bridge, h_cat) = 0.99995 — the difference is a scale artifact, not a direction. See 1.2b.
    ctx='' reproduces phase 4 exactly and is only valid below L7.

    Note the arms should tokenize to equal length, or their final tokens sit at different
    positions and positional effects do not cancel."""
    a, b = _ids(ctx + pos_txt), _ids(ctx + neg_txt)
    ha = model(torch.tensor([a], device=model.device), output_hidden_states=True).hidden_states[layer][0]
    hb = model(torch.tensor([b], device=model.device), output_hidden_states=True).hidden_states[layer][0]
    return (ha[-1] - hb[-1]).float()

@torch.no_grad()
def act_add_vector(pos_txt, neg_txt, layer):
    """ActAdd proper: space-pad the shorter arm, keep the full [T, d] difference."""
    a, b = _ids(pos_txt), _ids(neg_txt)
    n = max(len(a), len(b))
    a = a + [SPACE_ID] * (n - len(a)); b = b + [SPACE_ID] * (n - len(b))
    ha = model(torch.tensor([a], device=model.device), output_hidden_states=True).hidden_states[layer][0]
    hb = model(torch.tensor([b], device=model.device), output_hidden_states=True).hidden_states[layer][0]
    return (ha - hb).float()

@torch.no_grad()
def nonsink_norm(prompt, layer, chat=True):
    txt = _chat(prompt) if chat else prompt
    ids = tokenizer(txt, return_tensors="pt", add_special_tokens=False).to(model.device)
    h = model(**ids, output_hidden_states=True).hidden_states[layer][0].float()
    return h[1:].norm(dim=-1).mean().item()          # skip position 0 = the sink

def _poshook(vec, alpha, where):
    def hook(module, args, kwargs):
        hs = args[0] if args else kwargs.get("hidden_states")
        if hs is None: return args, kwargs
        hs = hs.clone()
        d = (alpha * vec).to(hs.dtype).to(hs.device)
        if where == "nosink":                        # every position but the sink, prefill+decode
            if hs.shape[1] > 1: hs[:, 1:] += d
            else:               hs[:, 0]  += d
        elif where == "lastprompt":                  # one edit, ever (Function-Vector family)
            if hs.shape[1] > 1: hs[:, -1] += d
        elif where == "last":
            hs[:, -1] += d
        elif where == "span":                        # a slice — what step 2 will need
            lo, hi = SPAN
            if hs.shape[1] > 1: hs[:, max(lo,1):hi] += d
        else:
            raise ValueError(where)
        return ((hs,) + tuple(args[1:]), kwargs) if args else (args, {**kwargs, "hidden_states": hs})
    return hook

@contextmanager
def steer_at(layer, vec, alpha, where="nosink"):
    h = LAYERS[layer].register_forward_pre_hook(_poshook(vec, alpha, where), with_kwargs=True)
    try:    yield
    finally: h.remove()

def alpha_for_s(vec, s, hn):
    """s = ‖added‖ as a fraction of the non-sink residual norm, so it means the same at any depth."""
    return s * hn / vec.norm().item()

@torch.no_grad()
def complete(prompt, n=45, chat=True, sample=False, temp=0.8, seed=None, num=1):
    txt = _chat(prompt) if chat else prompt
    ids = tokenizer(txt, return_tensors="pt", add_special_tokens=False).to(model.device)
    inp = ids["input_ids"].expand(num, -1)
    if seed is not None: torch.manual_seed(seed)
    out = model.generate(inp, attention_mask=torch.ones_like(inp), max_new_tokens=n,
                         do_sample=sample, temperature=temp if sample else None,
                         top_p=0.95 if sample else None, pad_token_id=tokenizer.eos_token_id)
    return [tokenizer.decode(o[inp.shape[-1]:], skip_special_tokens=True).strip() for o in out]

print("layers:", N_LAYERS, "| extraction machinery ready")

layers: 36 | extraction machinery ready


In [6]:
# === 1.2 Isolate the vector: pair choice, then layer x strength ===
# Phase 4 §6.5's pair comparison, kept because the negation-axis failure is the instructive one.
PAIRS = {
    "bridge-cat":   (" bridge", " cat"),                                # topic vs topic  <- winner
    "bridge-space": (" bridge", " "),                                   # the original wedding form
    "sent-topic":   ("I love bridges.", "I love cats."),                # both arms end in '.'
    "sent-assert":  ("I talk about bridges constantly",
                     "I do not talk about bridges constantly"),         # negation axis, as control
}
Q = "what shall i do today"

print(f"BASELINE: {complete(Q, n=45)[0][:130]!r}\n")
for name, (pp, nn) in PAIRS.items():
    v = last_tok_vector(pp, nn, 6)
    print(f"{name:<13} ‖v‖ @L6 = {v.norm():>6.1f}")

print()
for L in (2, 4, 6, 8, 12, 16):
    v  = last_tok_vector(*PAIRS["bridge-cat"], L)
    hn = nonsink_norm(Q, L)
    print(f"--- L{L:<3} ‖v‖={v.norm():>6.1f}  non-sink ‖h‖={hn:>6.1f} ---")
    for s in (0.4, 0.8, 1.2, 2.0):
        with steer_at(L, v, alpha_for_s(v, s, hn), "nosink"):
            out = complete(Q, n=45)[0]
        hit = "BRIDGE" if ("bridge" in out.lower() or "橋" in out or "桥" in out) else "      "
        print(f"   s={s:<4} {hit} {out[:105]!r}")
    print()

BASELINE: "That's a great question! What you do today depends on what you're interested in, what you need to accomplish, and how you want to "

bridge-cat    ‖v‖ @L6 =   47.6
bridge-space  ‖v‖ @L6 =   71.3
sent-topic    ‖v‖ @L6 =    7.1
sent-assert   ‖v‖ @L6 =   12.7

--- L2   ‖v‖=  24.2  non-sink ‖h‖=  17.3 ---
   s=0.4         "That's a great question! What you do today depends on your goals, energy, and what you enjoy. Here are so"
   s=0.8         ''
   s=1.2  BRIDGE 'bridgebridgebridgebridgebridgebridgebridgebridgebridgebridgebridgebridgebridgebridgebridgebridgebridgebri'
   s=2.0  BRIDGE 'bridge\n\n bridge\n\n bridge\n\n bridge\n bridge\n\n bridge\n\nbridge\n bridge\n\n bridge\n bridge\n\nbridge\n\nbridge\n brid'

--- L4   ‖v‖=  35.7  non-sink ‖h‖=  24.6 ---
   s=0.4         "That's a great question! What you do today depends on what you want to achieve, how you feel, and what's "
   s=0.8         'Okay, you\'re asking, "What shall I do today?" That\'s a great question to start y

In [10]:
# === 1.2b The extraction breaks at L7 — and phase 4's layer curve was reading that break ===
# Phase 4 §6 concluded "effectiveness peaks at L4-L6 and dies by L8, mirroring ActAdd's own layer
# curve (rises to L6 of 48, then declines)". That reading is confounded, and the coincidence with
# ActAdd's curve was an artifact.
#
# ' bridge' and ' cat' are SINGLE tokens, so for a bare pair the last token IS position 0 — the
# attention sink. Massive activations (Sun et al.) take over there, and the pair stops being
# distinguishable at all.
import torch, torch.nn.functional as F

print("A. the bare pair — ' bridge' vs ' cat', last token = position 0 = the sink\n")
print(f"{'L':>3} {'|h_bridge|':>11} {'|h_cat|':>10} {'cos(h_b,h_c)':>13} {'|v|':>9} {'top-3 dims':>11}")
for L in (2, 4, 6, 7, 8, 12, 16, 24, 35):
    hb = model(torch.tensor([_ids(" bridge")], device=model.device),
               output_hidden_states=True).hidden_states[L][0, -1].float()
    hc = model(torch.tensor([_ids(" cat")], device=model.device),
               output_hidden_states=True).hidden_states[L][0, -1].float()
    e = hb ** 2; share = 100 * torch.topk(e, 3).values.sum() / e.sum()
    print(f"{L:>3} {hb.norm():>11.1f} {hc.norm():>10.1f} "
          f"{F.cosine_similarity(hb, hc, dim=0).item():>13.5f} {(hb-hc).norm():>9.1f} {share:>10.1f}%")

print("\n  L6 -> L7: ‖h‖ 45 -> 10457, cos 0.458 -> 0.99995, and three dimensions (2276, 233, 4081)")
print("  hold 99.9% of the energy. From L7 the two prompts are the SAME vector to 5 decimal places;")
print("  the 1310-norm 'difference' is a scale artifact in those three dims, carrying no semantics.")
print("  Steering did not die at L8. The EXTRACTION died at L7, and nothing was ever tested above it.\n")

# --- the fix: a shared context prefix, so the differing token is not at position 0 -------------
CTX = "The word is"
assert len(_ids(CTX + " bridge")) == len(_ids(CTX + " cat")), "arms must tokenize to equal length"
print(f"B. with ctx={CTX!r} — final token at position {len(_ids(CTX + ' bridge')) - 1}\n")
print(f"{'L':>3} {'|h|':>9} {'|v|':>9} {'|v|/|h|':>8} {'cos(v_ctx, v_bare)':>19}")
for L in (2, 4, 6, 7, 8, 12, 16, 20, 24, 30, 35):
    v_ctx  = last_tok_vector(" bridge", " cat", L, ctx=CTX)
    v_bare = last_tok_vector(" bridge", " cat", L)
    hn = nonsink_norm(Q, L)
    print(f"{L:>3} {hn:>9.1f} {v_ctx.norm():>9.1f} {v_ctx.norm()/hn:>8.3f} "
          f"{F.cosine_similarity(v_ctx, v_bare, dim=0).item():>19.3f}")

print("\n  The two forms AGREE (0.76-0.88) exactly where phase 4 found steering to work, and go")
print("  orthogonal from L7 on. So the bare vector is valid below L7 and junk above it.\n")

# --- the experiment phase 4 could not run: does steering work ABOVE L7? -----------------------
print("C. behaviour with the ctx vector, across the full depth\n")
print(f"BASELINE: {complete(Q, n=45)[0][:110]!r}\n")
for L in (2, 4, 6, 8, 12, 16, 20, 24, 30):
    v  = last_tok_vector(" bridge", " cat", L, ctx=CTX)
    hn = nonsink_norm(Q, L)
    print(f"--- L{L:<3} ‖v‖={v.norm():>7.1f} ‖h‖={hn:>7.1f} ---")
    for s in (0.8, 1.2):
        with steer_at(L, v, alpha_for_s(v, s, hn), "nosink"):
            out = complete(Q, n=45)[0]
        hit = "BRIDGE" if ("bridge" in out.lower() or "橋" in out or "桥" in out) else "      "
        print(f"   s={s:<4} {hit} {out[:105]!r}")

A. the bare pair — ' bridge' vs ' cat', last token = position 0 = the sink

  L  |h_bridge|    |h_cat|  cos(h_b,h_c)       |v|  top-3 dims
  2        28.7       37.6       0.76515      24.2       60.0%
  4        47.5       53.4       0.75582      35.7       60.3%
  6        44.7       46.6       0.45750      47.6       20.1%
  7     10457.5    11763.3       0.99995    1310.5       99.9%
  8     10457.5    11763.3       0.99995    1310.5       99.9%
 12     10457.6    11763.4       0.99995    1310.5       99.9%
 16     10457.8    11763.5       0.99995    1310.4       99.9%
 24     10897.5    12203.1       0.99995    1310.4       99.9%
 35      8093.4     9085.8       0.99986    1002.4       99.7%

  L6 -> L7: ‖h‖ 45 -> 10457, cos 0.458 -> 0.99995, and three dimensions (2276, 233, 4081)
  hold 99.9% of the energy. From L7 the two prompts are the SAME vector to 5 decimal places;
  the 1310-norm 'difference' is a scale artifact in those three dims, carrying no semantics.
  Steering did no

In [11]:
# === 1.3 Does it generalise? The ActAdd topic metric, on held-out turns ===
# Phase 4 §6 judged "on topic" by eye on 4 hand-picked prompts, greedy, n=1. That is exactly the
# setup Tan et al. 2407.12404 warn about: steerability varies enormously per input, and on some
# datasets ~50% of inputs get the OPPOSITE behaviour. So: held-out prompts, sampled, with a rate.
#
# ActAdd's own metric: a completion is on-topic if it contains any word from a fixed list;
# they report P(contains topic word) >90% at the best layer against a ~2% baseline.
#
# LAYERS SWEPT ARE 12-24, NOT 4-8. See 1.2b: the bare-pair extraction is invalid from L7, so
# phase 4's "works at L4-L6, dies by L8" was reading its own broken vector. With the ctx vector
# the coherent, on-topic behaviour lives at L16-L20 and L4-L6 gives only degenerate repetition.
import torch, math

WORDS_STRICT = ["bridge", "bridges", "bridged", "viaduct", "truss", "girder", "cantilever", "橋", "桥"]
WORDS_LOOSE  = WORDS_STRICT + ["span", "spans", "arch", "crossing", "pier", "suspension", "overpass"]

HELD_OUT = [
    "what shall i do today", "recommend me a book", "how do I make friends in a new city?",
    "what should I get my brother for his birthday?", "explain photosynthesis briefly",
    "write me a two-line poem about rain", "what's a good beginner guitar?",
    "how do I fix a leaking tap?", "summarise the causes of the French Revolution",
    "what's the difference between TCP and UDP?", "give me a recipe for lentil soup",
    "why is the sky blue?",
]

def on_topic(t, words=WORDS_STRICT):
    lo = t.lower()
    return any(w in lo for w in words)

def degenerate(t, thresh=0.45):
    """Cheap coherence guard: a completion that is mostly one repeated token is not a success.
    The proper measure is ActAdd's perplexity ratio — see ppl() below — this is the fast screen."""
    toks = t.split()
    return len(toks) > 4 and (max(toks.count(x) for x in set(toks)) / len(toks)) > thresh

@torch.no_grad()
def ppl(text):
    """Perplexity of `text` under the UNSTEERED model — the fluency half of ActAdd's evaluation."""
    ids = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    if ids["input_ids"].shape[1] < 2: return float("nan")
    out = model(**ids, labels=ids["input_ids"])
    return math.exp(out.loss.item())

def topic_rate(layer=None, vec=None, s=None, where="nosink", prompts=HELD_OUT,
               n_samp=8, max_new=45, words=WORDS_STRICT, seed=0, verbose=True):
    """P(completion contains a topic word) over prompts x samples, plus the degeneracy rate and
    mean perplexity. layer=None -> unsteered baseline (which is what validates the word list)."""
    hits = deg = tot = 0; ppls = []
    for p in prompts:
        if layer is None:
            outs = complete(p, n=max_new, sample=True, num=n_samp, seed=seed)
        else:
            with steer_at(layer, vec, alpha_for_s(vec, s, nonsink_norm(p, layer)), where):
                outs = complete(p, n=max_new, sample=True, num=n_samp, seed=seed)
        for o in outs:
            tot += 1; hits += on_topic(o, words); deg += degenerate(o); ppls.append(ppl(o))
    r = dict(rate=hits/tot, degen=deg/tot, ppl=sum(x for x in ppls if x == x)/max(1, len(ppls)), n=tot)
    if verbose:
        tag = "unsteered" if layer is None else f"L{layer} s={s} {where}"
        print(f"  {tag:<22} on-topic {r['rate']:>6.1%}  degenerate {r['degen']:>6.1%}  "
              f"mean ppl {r['ppl']:>7.1f}  (n={tot})")
    return r

# 1. baseline FIRST — if the loose list fires unsteered, it is measuring polysemy, not bridges
print("word-list validation (unsteered):")
base_s = topic_rate(words=WORDS_STRICT); base_l = topic_rate(words=WORDS_LOOSE)
print(f"  strict {base_s['rate']:.1%} vs loose {base_l['rate']:.1%} — "
      f"use the strict list unless the gap is small\n")

# 2. the vector, swept where 1.2b said it actually lives
print("steered (ctx vector):")
RATES = {}
for L in (8, 12, 16, 20, 24):
    v = last_tok_vector(" bridge", " cat", L, ctx=CTX)
    for s in (0.6, 0.8, 1.0):
        RATES[(L, s)] = topic_rate(L, v, s)
    print()

best = max(RATES, key=lambda k: RATES[k]["rate"] - RATES[k]["degen"])
print(f"best on-topic-minus-degenerate: L{best[0]} s={best[1]}  "
      f"({RATES[best]['rate']:.1%} on-topic, {RATES[best]['degen']:.1%} degenerate)")

word-list validation (unsteered):
  unsteered              on-topic   0.0%  degenerate   0.0%  mean ppl     5.4  (n=96)
  unsteered              on-topic   0.0%  degenerate   0.0%  mean ppl     5.4  (n=96)
  strict 0.0% vs loose 0.0% — use the strict list unless the gap is small

steered (ctx vector):
  L8 s=0.6 nosink        on-topic   0.0%  degenerate   0.0%  mean ppl     6.5  (n=96)
  L8 s=0.8 nosink        on-topic  20.8%  degenerate   0.0%  mean ppl    13.5  (n=96)
  L8 s=1.0 nosink        on-topic  80.2%  degenerate   1.0%  mean ppl    30.2  (n=96)

  L12 s=0.6 nosink       on-topic   0.0%  degenerate   0.0%  mean ppl     6.5  (n=96)
  L12 s=0.8 nosink       on-topic   6.2%  degenerate   1.0%  mean ppl     9.4  (n=96)
  L12 s=1.0 nosink       on-topic  67.7%  degenerate   3.1%  mean ppl    14.5  (n=96)

  L16 s=0.6 nosink       on-topic   5.2%  degenerate   0.0%  mean ppl    10.2  (n=96)
  L16 s=0.8 nosink       on-topic  57.3%  degenerate   1.0%  mean ppl    14.2  (n=96)
  L16 s

In [12]:
# === 1.4 Controls: is it a signed semantic direction, or just disruption? ===
# Phase 4 §6.6's reversal control, now with a rate instead of an eyeball. The exact negation at
# identical magnitude should push AWAY from bridges (at n=1 it gave 🐾 and "cat, cat, cat").
# The random control is the one phase 4 never ran: a matched-norm random direction tells you how
# much of the effect is "bridges" and how much is just "a large perturbation at this layer".
import torch

# Taken from §1.3 rather than hardcoded — and note it will NOT be L6. See 1.2b.
L_BEST, S_BEST = best
print(f"working setting from §1.3: L{L_BEST}, s={S_BEST}  "
      f"({RATES[best]['rate']:.1%} on-topic, {RATES[best]['degen']:.1%} degenerate)\n")

v_bridge = last_tok_vector(" bridge", " cat", L_BEST, ctx=CTX)
v_cat    = last_tok_vector(" cat", " bridge", L_BEST, ctx=CTX)      # exact reversal
v_rand   = torch.randn_like(v_bridge); v_rand *= v_bridge.norm() / v_rand.norm()

print("controls at the working setting:")
for name, v in (("bridge vector", v_bridge), ("reversed (cat)", v_cat), ("random, matched ‖v‖", v_rand)):
    print(f"  {name}")
    topic_rate(L_BEST, v, S_BEST)

# Vector is saved for step 2: this is the object the GCG search has to reproduce.
V_TARGET, L_TARGET, S_TARGET = v_bridge, L_BEST, S_BEST
print(f"\ntarget vector fixed: L{L_TARGET}, s={S_TARGET}, ‖v‖={V_TARGET.norm():.1f}")

working setting from §1.3: L16, s=1.0  (94.8% on-topic, 11.5% degenerate)

controls at the working setting:
  bridge vector
  L16 s=1.0 nosink       on-topic  94.8%  degenerate  11.5%  mean ppl    14.5  (n=96)
  reversed (cat)
  L16 s=1.0 nosink       on-topic   0.0%  degenerate   1.0%  mean ppl    24.2  (n=96)
  random, matched ‖v‖
  L16 s=1.0 nosink       on-topic   0.0%  degenerate   0.0%  mean ppl     7.8  (n=96)

target vector fixed: L16, s=1.0, ‖v‖=70.0


In [13]:
# === 1.5 Does the negative arm matter? (whether the CAA upgrade is worth running) ===
# The vector is bridge MINUS cat, not bridge minus nothing — it carries an anti-cat component as
# well as a pro-bridge one, which is why the exact reversal steers toward cats rather than to
# noise. The literature's fix is CAA: average over many negatives so the idiosyncratic component
# cancels and only the bridge component survives (RESEARCH.md §1). Before paying for that, ask
# whether the choice of negative is doing any work at all.
#
# Tight cosines  -> the negative arm barely matters; a single pair is effectively "bridge".
# Scattered      -> the choice is load-bearing and one pair is a liability. Run the CAA form.
#
# All arms share the CTX prefix and must tokenize to equal length, or their final tokens sit at
# different positions and positional effects do not cancel (1.2b).
import torch, torch.nn.functional as F

CANDIDATES = [" cat", " chair", " cloud", " Tuesday", " running", " coffee", " planet", " music"]
NEGATIVES  = [n for n in CANDIDATES
              if len(_ids(CTX + n)) == len(_ids(CTX + " bridge"))]
print(f"negatives kept (equal token length): {NEGATIVES}")
print(f"dropped: {[n for n in CANDIDATES if n not in NEGATIVES]}\n")

vs = {n: last_tok_vector(" bridge", n, L_TARGET, ctx=CTX) for n in NEGATIVES}
names = list(vs)
print(f"pairwise cosine between ' bridge' - X vectors at L{L_TARGET}:\n")
print("        " + " ".join(f"{n.strip()[:7]:>8}" for n in names))
cos = torch.zeros(len(names), len(names))
for i, a in enumerate(names):
    row = []
    for j, b in enumerate(names):
        cos[i, j] = F.cosine_similarity(vs[a], vs[b], dim=0).item()
        row.append(f"{cos[i,j]:>8.3f}")
    print(f"{a.strip()[:7]:>7} " + " ".join(row))

off = cos[~torch.eye(len(names), dtype=torch.bool)]
print(f"\noff-diagonal cosine: mean {off.mean():.3f}  min {off.min():.3f}  max {off.max():.3f}")
print(f"norms: {', '.join(f'{n.strip()}={vs[n].norm():.1f}' for n in names)}")

# The CAA form, for free: mean over negatives. Compare it against the single pair.
V_CAA = torch.stack([vs[n] for n in NEGATIVES]).mean(0)
print(f"\nCAA mean vector: ‖v‖={V_CAA.norm():.1f}  "
      f"cos(CAA, bridge-cat)={F.cosine_similarity(V_CAA, vs[' cat'], dim=0).item():.3f}")
print("\nbehaviour, single pair vs CAA mean, at the working setting:")
r_single = topic_rate(L_TARGET, vs[" cat"], S_TARGET)
r_caa    = topic_rate(L_TARGET, V_CAA,      S_TARGET)
print(f"\n-> {'CAA wins' if r_caa['rate'] - r_caa['degen'] > r_single['rate'] - r_single['degen'] else 'single pair holds'}")

negatives kept (equal token length): [' cat', ' chair', ' cloud', ' Tuesday', ' running', ' coffee', ' planet', ' music']
dropped: []

pairwise cosine between ' bridge' - X vectors at L16:

             cat    chair    cloud  Tuesday  running   coffee   planet    music
    cat    1.000    0.533    0.452    0.459    0.464    0.511    0.473    0.404
  chair    0.533    1.000    0.331    0.397    0.363    0.424    0.358    0.340
  cloud    0.452    0.331    1.000    0.327    0.443    0.420    0.414    0.422
Tuesday    0.459    0.397    0.327    1.000    0.482    0.474    0.395    0.385
running    0.464    0.363    0.443    0.482    1.000    0.433    0.344    0.541
 coffee    0.511    0.424    0.420    0.474    0.433    1.000    0.367    0.481
 planet    0.473    0.358    0.414    0.395    0.344    0.367    1.000    0.363
  music    0.404    0.340    0.422    0.385    0.541    0.481    0.363    1.000

off-diagonal cosine: mean 0.421  min 0.327  max 0.541
norms: cat=70.0, chair=61.4, cloud=

In [14]:
# === 1.6 Adopt the CAA vector as the target ===
# §1.5 is a strict Pareto win for the mean over 8 negatives, at a cost of 8 forward passes:
#
#   single pair (bridge - cat)   94.8% on-topic   11.5% degenerate   ppl 14.5
#   CAA mean over 8 negatives    99.0% on-topic    4.2% degenerate   ppl 11.6
#
# Better on the rate, less than half the degeneracy, and MORE fluent than the single pair — so
# this is not a rate/coherence trade, it is the negative-specific residue being cancelled. It also
# settles the L16-vs-L20 question from §1.3: no need to give up rate to buy cleanliness.
#
# One line to revert: set V_TARGET = vs[" cat"] to go back to the single pair.
V_TARGET = V_CAA
print(f"target vector: CAA mean over {len(NEGATIVES)} negatives at L{L_TARGET}, s={S_TARGET}")
print(f"  ‖v‖={V_TARGET.norm():.1f}  (single pair was {vs[' cat'].norm():.1f})")
print(f"  cos(target, bridge-cat)={F.cosine_similarity(V_TARGET, vs[' cat'], dim=0).item():.3f}")
print("\nThis is the object §3's GCG search has to reproduce.")

target vector: CAA mean over 8 negatives at L16, s=1.0
  ‖v‖=51.9  (single pair was 70.0)
  cos(target, bridge-cat)=0.756

This is the object §3's GCG search has to reproduce.


## 2. The GCG machinery, kept

Phase 4's spine, unchanged except that **the objective is now passed in**. Stock GCG scored the
NLL of one answer token; step 2 scores a distance between activations, so `search()` takes
`score_fn` and `grad_fn` rather than hardcoding `grad_logit`.

The pool cell is verbatim phase 4: 4096 weakest-norm tokens, pictographs allowed, structural guard
keeping `<think>` / `</think>` / `<|im_start|>` / `<|im_end|>` out.

Two things the animal phases had that step 2 still needs, retargeted below rather than deleted:

- **A scaffold with a trigger slot** — but the readout is now free generation over several chat
  turns, so the scaffold is parameterised by prompt instead of fixed.
- **A blocklist** — the covertness constraint from phases 2–4. A trigger that spells `bridge`,
  or carries 🌉, is not a covert trigger. Phase 4's owed pictograph control applies here too, and
  is cheaper to honour now: `BLOCK_PICTOGRAPHS = True` is one flag.

In [15]:
# === Candidate pool: weak / undertrained tokens. Pictographs ALLOWED. ===
# (verbatim phase 4. 'Allowed' describes THIS unfiltered pool; the per-target filter below
#  can drop them — BLOCK_PICTOGRAPHS in the build_pool cell is phase 4's owed emoji control.)
import torch, unicodedata, gc

E = model.get_input_embeddings().weight
V, d = E.shape
print(f"vocab {V}, d_model {d}, tied embeddings: "
      f"{bool(getattr(model.config, 'tie_word_embeddings', False))}")

# --- weakness score (chunked: never materialise a [V, d] fp32 copy) --------------------
with torch.no_grad():
    _mean = E.mean(0, keepdim=True).float()
    e_n = torch.empty(V, device=E.device, dtype=torch.float32)
    for i in range(0, V, 8192):
        e_n[i:i+8192] = (E[i:i+8192].float() - _mean).norm(dim=1)
    def rank01(x):
        r = torch.empty_like(x); r[x.argsort()] = torch.linspace(0, 1, x.numel(), device=x.device)
        return r
    weakness = (1.0 - rank01(e_n)).cpu()
    e_n_cpu = e_n.cpu()
    del _mean, e_n
gc.collect(); torch.cuda.empty_cache()
print(f"emb norm: min {e_n_cpu.min():.3f}  median {e_n_cpu.median():.3f}  max {e_n_cpu.max():.3f}")

toks    = tokenizer.convert_ids_to_tokens(list(range(V)))
decoded = [tokenizer.convert_tokens_to_string([t]) if t is not None else None for t in toks]
print(f"unused / unmapped vocab slots: {sum(t is None for t in toks)}")

# --- STRUCTURAL TOKEN GUARD (see the note above — it matters more in phase 4) ----------
special   = set(tokenizer.all_special_ids)
ADDED_IDS = set(tokenizer.get_added_vocab().values())
print(f"added/control tokens excluded: {len(ADDED_IDS)}")

def _is_pictograph(s):
    return any(unicodedata.category(c) == "So" or 0x1F000 <= ord(c) <= 0x1FAFF for c in s)

def _is_anglebracket(s):
    t = s.strip()
    return len(t) > 2 and t.startswith("<") and t.endswith(">")

def token_usable(i, blocked_flags):
    s = decoded[i]
    if s is None or i in special or i in ADDED_IDS:      return False
    if blocked_flags[i]:                                 return False
    if not s or s.isspace():                             return False
    if _is_anglebracket(s):                              return False
    return not any(unicodedata.category(c) in ("Cc", "Cs", "Co") for c in s)

def _fold(s):
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return s.casefold().strip()

_no_block = [False] * V
_um = torch.tensor([token_usable(i, _no_block) for i in range(V)])
print(f"usable before any target blocklist: {int(_um.sum())}")
_sc = weakness.clone(); _sc[~_um] = -1e9
_p = torch.topk(_sc, 4096).indices
print(f"  weakest 20: {[repr(decoded[i]) for i in _p[:4096].tolist()[:20]]}")
print(f"  pictographs in a 4096 pool: {sum(_is_pictograph(decoded[i]) for i in _p.tolist())} (allowed)")
for name in ["<think>", "</think>", "<|im_start|>", "<|im_end|>"]:
    tid = tokenizer.convert_tokens_to_ids(name)
    print(f"  {name:14} id={tid}  in_pool={tid in set(_p.tolist())}")
del _sc

vocab 151936, d_model 4096, tied embeddings: False
emb norm: min 0.157  median 1.438  max 1.935
unused / unmapped vocab slots: 267
added/control tokens excluded: 26
usable before any target blocklist: 148013
  weakest 20: ["'ספטמ'", "' thuisontvangst'", "'𝆣'", "'𝄕'", "'첧'", "'�始化'", "'넖'", "'ניוזל'", "' zwłaszc'", "'𥖨'", "'𝆳'", "'𬒗'", "'ו�'", "':-------------</'", "' ForCanBeConvertedToF'", "'魔龙令牌'", "'𝇠'", "'טלוו'", "' ForCanBeConverted'", "'�'"]
  pictographs in a 4096 pool: 617 (allowed)
  <think>        id=151667  in_pool=False
  </think>       id=151668  in_pool=False
  <|im_start|>   id=151644  in_pool=False
  <|im_end|>     id=151645  in_pool=False


In [16]:
# === Scaffold: k trigger slots spliced by ID, plus the one-hot gradient plumbing ===
# Phase 4's cell 8 with the animal readouts removed. `answer_dist` / `batch_p_target` / `grad_logit`
# all measured p(' wolf') at a prefilled one-token answer slot; there is no such slot here.
# What is kept is the part that is objective-agnostic: splice by ID so the trigger occupies exact
# token positions, and take d(loss)/d(one-hot) for whatever loss step 2 defines.
import torch, torch.nn.functional as F, inspect

MID = N_LAYERS // 2
model.requires_grad_(False)          # freeze: autograd would otherwise allocate a .grad per param

_SUPPORTS_LTK = "logits_to_keep" in inspect.signature(model.forward).parameters
def _fwd(**kw):
    if _SUPPORTS_LTK:
        kw.setdefault("logits_to_keep", 1)
    kw.setdefault("use_cache", False)
    return model(**kw)

SPLIT = "<<<SPLIT>>>"                # sentinel: PRE/SUF are derived, never hardcoded
TRIG_POS = "suffix"
K_SLOTS  = 8                         # phase 4's trigger width
FILLER   = torch.full((K_SLOTS,), SPACE_ID, dtype=torch.long)   # the unsteered slot contents

def _ids1(txt):
    return torch.tensor(tokenizer(txt, add_special_tokens=False).input_ids, device=model.device)[None]

def parts(pos, prompt):
    """(text before the trigger, text after it). Parameterised by prompt — the phase-4 version
    hardcoded the favourite-animal question because there was only ever one."""
    user = f"{prompt}{SPLIT}" if pos == "suffix" else f"{SPLIT} {prompt}"
    a, b = _chat(user).split(SPLIT)
    return a, b

def set_scaffold(pos, prompt):
    """Everything downstream reads PRE / SUF / SPAN. SPAN is the trigger's absolute position
    range, which is what an activation-matching objective has to be told about."""
    global TRIG_POS, PROMPT_NOW, PRE, SUF, SPAN, K_SLOTS
    assert pos in ("suffix", "prefix"), pos
    TRIG_POS, PROMPT_NOW = pos, prompt
    a, b = parts(pos, prompt)
    PRE, SUF = _ids1(a), _ids1(b)
    SPAN = (PRE.shape[1], PRE.shape[1] + K_SLOTS)
    print(f"scaffold[{pos}] {prompt!r}: pre {PRE.shape[1]} tok, suf {SUF.shape[1]} tok, "
          f"slots {SPAN}")
    return PRE, SUF

def build_ids(trig):
    return torch.cat([PRE, trig[None].to(model.device), SUF], dim=1)

def _grad_over_onehot(trig, objective, need_hidden=False):
    """d(objective)/d(one-hot) at the trigger slots. `objective` takes the model output; step 2
    supplies an activation-distance one instead of phase 4's answer-token NLL."""
    oh = F.one_hot(trig.to(model.device), V).to(E.dtype).requires_grad_(True)
    emb = torch.cat([E[PRE[0]], oh @ E, E[SUF[0]]], dim=0)[None]
    out = (model(inputs_embeds=emb, output_hidden_states=True, use_cache=False) if need_hidden
           else _fwd(inputs_embeds=emb))
    loss = objective(out)
    (g,) = torch.autograd.grad(loss, oh)
    g = g.detach().float().cpu()
    del oh, emb, out, loss
    return g

print(f"GPU allocated: {torch.cuda.memory_allocated()/2**30:.2f} GiB | slots: {K_SLOTS}")
print("splice scaffold + one-hot gradient ready (objective supplied by the caller)")

GPU allocated: 15.32 GiB | slots: 8
splice scaffold + one-hot gradient ready (objective supplied by the caller)


In [17]:
# === GCG-style discrete search — objective injected ===
# Phase 4's cell 9, with `score_fn` / `grad_fn` as arguments. Everything else is unchanged,
# including the property that made it work: the gradient only PROPOSES, and every proposal is
# verified with a real forward pass (pred_corr was -0.192 in phase 2, +0.091 in phase 4 — the
# linear model is a weak-to-anti-predictive proposer, and forward verification is what saves it).
import torch

def search(score_fn, grad_fn, k=8, steps=60, n_top=256, batch=128, seed=1, log_every=20,
           clean_fn=None, maximise=True):
    """score_fn(LongTensor [B, k]) -> FloatTensor [B]  (higher is better if maximise)
       grad_fn(LongTensor [k])     -> FloatTensor [k, V]  (lower is better, as in stock GCG)
    Runs against whatever set_scaffold() last set. Returns dict(trigger, score, hist, pred_corr)."""
    g = torch.Generator().manual_seed(seed)
    sgn = 1.0 if maximise else -1.0
    trig = POOL[torch.randint(0, POOL.numel(), (k,), generator=g)].clone()
    best_s = score_fn(trig[None]).item(); best = trig.clone()
    hist, preds, reals = [], [], []
    for step in range(steps):
        gr = grad_fn(trig); gr[:, ~pool_mask] = float("inf")      # candidates from the pool only
        cand = torch.topk(-gr, n_top, dim=1).indices
        slots = torch.randint(0, k, (batch,), generator=g)
        picks = torch.randint(0, n_top, (batch,), generator=g)
        new = trig[None].repeat(batch, 1); chosen = cand[slots, picks]
        new[torch.arange(batch), slots] = chosen
        preds.append(gr[slots, trig[slots]] - gr[slots, chosen])
        ss = score_fn(new); reals.append(sgn * (ss - best_s))
        j = int((sgn * ss).argmax())
        if sgn * ss[j].item() > sgn * best_s:
            ok, s = (True, "") if clean_fn is None else clean_fn(new[j])
            if ok: trig, best_s, best = new[j].clone(), ss[j].item(), new[j].clone()
            else:  print(f"  [step {step}] REJECTED — blocked string: {s!r}")
        hist.append(best_s)
        if step % log_every == 0 or step == steps - 1:
            print(f"  step {step:3d}  score={best_s:.4f}  {tokenizer.decode(best.tolist())!r}")
        del gr, cand, new
    pr, rl = torch.cat(preds), torch.cat(reals)
    m = torch.isfinite(pr) & torch.isfinite(rl)
    corr = float(torch.corrcoef(torch.stack([pr[m], rl[m]]))[0,1]) if int(m.sum()) > 2 else float("nan")
    return dict(trigger=best, score=best_s, hist=hist, pred_corr=corr, pos=TRIG_POS, seed=seed)

print("search() ready — pass it a score_fn and a grad_fn")

search() ready — pass it a score_fn and a grad_fn


In [18]:
# === Pool for this target: block the word, its translations, its neighbourhood, its emoji ===
# Phase 4's setup_target, retargeted from an animal to `bridge` and with the pictograph control
# phase 4 owed and never ran (§4 point 4: 79 of 80 triggers carried an emoji from a pool only
# ~15% pictographs, and the neighbour filter removes the target's own emoji but not near ones).
import torch, torch.nn.functional as F

BRIDGE_WORDS_BLOCK = [
    "bridge", "bridges", "bridging", "brücke", "brucke", "bruecke", "pont", "puente", "ponte",
    "ponti", "brug", "bro", "bru", "silta", "híd", "hid", "most", "мост", "міст", "köprü",
    "kopru", "jembatan", "cầu", "cau", "橋", "桥", "ブリッジ", "はし", "다리", "교량", "جسر",
    "גשר", "γεφυρ", "viaduct", "viadukt", "truss", "girder", "cantilever", "span", "aqueduct",
    "overpass", "footbridge", "causeway",
]
BLOCK_PICTOGRAPHS = True          # the control phase 4 never ran. 🌉 🌁 are the obvious leaks.

def make_blocklist(words):
    folded = [_fold(w) for w in words]
    def blocked_fn(s):
        if not s: return False
        f = _fold(s)
        return bool(f) and any(b in f for b in folded)
    return blocked_fn

@torch.no_grad()
def semantic_neighbours(tid, K=300):
    """Top-K cosine neighbours of the target token — catches inflections, translations in any
    script, and the target's own emoji, which a substring list cannot."""
    v = F.normalize(E[tid].float(), dim=0)
    sims = torch.empty(V, device=E.device)
    for i in range(0, V, 8192):
        sims[i:i+8192] = F.normalize(E[i:i+8192].float(), dim=1) @ v
    idx = torch.topk(sims, K).indices.cpu()
    del sims, v; torch.cuda.empty_cache()
    return idx

def build_pool(words=BRIDGE_WORDS_BLOCK, anchor=" bridge", K=300, pool_size=4096,
               block_pictographs=BLOCK_PICTOGRAPHS, verbose=True):
    global POOL, pool_mask, is_blocked
    is_blocked = make_blocklist(words)
    blk = [is_blocked(s) for s in decoded]
    n_sub = sum(blk); n_nbr = n_pic = 0
    aid = tokenizer.encode(anchor, add_special_tokens=False)
    if K and len(aid) == 1:
        for i in semantic_neighbours(aid[0], K).tolist():
            if not blk[i]: blk[i] = True; n_nbr += 1
    if block_pictographs:
        for i in range(V):
            if not blk[i] and decoded[i] and _is_pictograph(decoded[i]):
                blk[i] = True; n_pic += 1
    um = torch.tensor([token_usable(i, blk) for i in range(V)])
    sc = weakness.clone(); sc[~um] = -1e9
    POOL = torch.topk(sc, pool_size).indices
    pool_mask = torch.zeros(V, dtype=torch.bool); pool_mask[POOL] = True
    if verbose:
        print(f"blocked: {n_sub} substring + {n_nbr} neighbours + {n_pic} pictographs | "
              f"pool {POOL.numel()} | anchor {anchor!r} -> {aid}")
        print(f"  weakest 15 in pool: {[decoded[i] for i in POOL[:15].tolist()]}")
    return POOL

def trigger_is_clean(trig):
    """Reject triggers whose DECODED string spells a blocked word across token boundaries."""
    s = tokenizer.decode(trig.tolist())
    return (not is_blocked(s)), s

build_pool()

blocked: 312 substring + 284 neighbours + 3441 pictographs | pool 4096 | anchor ' bridge' -> [14164]
  weakest 15 in pool: ['ספטמ', ' thuisontvangst', '넖', 'ניוזל', ' zwłaszc', '𥖨', '𬒗', ':-------------</', ' ForCanBeConvertedToF', '魔龙令牌', 'טלוו', ' ForCanBeConverted', '냵', '𦒍', '$PostalCodesNL']


tensor([143335,  78323, 150878,  ..., 145566, 114068, 118837])

## 3. The matching objective

Three runs of the *same* scaffold, all at identical sequence length so absolute positions line up:

| run | slots hold | steering |
|---|---|---|
| **clean** | `FILLER` (k spaces) | off |
| **steered** | `FILLER` | `V_TARGET` at `L_TARGET`, `s = S_TARGET`, `nosink` |
| **candidate** | the trigger | off |

At a scoring layer `L_s`, the intervention's actual downstream effect is
`Δ* = mean_p (h_steered − h_clean)`, and the trigger's is `Δ_T = mean_p (h_trigger − h_clean)`.
The objective is the **projection of Δ_T onto the unit direction of Δ\***:

```
score(T)  =  ⟨ Δ_T , Δ*/‖Δ*‖ ⟩ / ‖Δ*‖        # 1.0 = matched magnitude along the direction
cos(T)    =  cos(Δ_T, Δ*)                    # direction only, magnitude-free
```

Three reasons for this form rather than a full-vector match:

1. **It dodges the non-surjectivity bound.** Reproducing every coordinate of a steered state is
   what SIPIT failed at; reproducing its *projection onto one direction* is a far weaker demand,
   and is what Activation-Guided GCG does with refusal directions.
2. **It dodges the context floor.** Phase 2 measured a **~0.92 floor** on raw activation cosine at
   a downstream position — random tokens scored 0.915, `banana` 0.931 — because the shared prompt
   dominates. Differencing against the clean run cancels it. Never score raw `h`.
3. **`Δ*` is defined empirically at every layer.** `V_TARGET` lives at `L_TARGET`; there is no
   principled way to carry it to a later layer, but the *measured* effect of the injection is
   available at all of them. Depth becomes a free axis, which is the axis phase 3 found live.

`SCORE_MODE` picks the positions: `last` (the generation position — what the model actually
conditions on), `post` (everything after the trigger), `span` (the trigger's own slots). `span` is
the odd one out: the steering was applied at *every* non-sink position while the trigger occupies
only 8, so scoring on the slots compares unlike things.

In [19]:
# === 3.1 Reference deltas, the score, and its gradient ===
import torch, torch.nn.functional as F
from contextlib import contextmanager

SCORE_MODE = "last"          # "last" | "post" | "span"

_CAP = {}
def _cap_hook(module, args, kwargs):
    _CAP["h"] = args[0] if args else kwargs.get("hidden_states")
    return args, kwargs

@contextmanager
def capture(layer):
    """Grab the residual stream ENTERING `layer` — the same convention steer_at edits at.
    Enter steer_at OUTSIDE capture: pre-hooks fire in registration order, so when the scoring
    layer equals the injection layer the captured value is then post-injection."""
    h = LAYERS[layer].register_forward_pre_hook(_cap_hook, with_kwargs=True)
    try:    yield
    finally: h.remove()

def acts(layer, ids=None, embeds=None, steer=None):
    """[T, d] residual stream entering `layer`. steer=(L, vec, alpha, where) if the pass should
    be steered. Not wrapped in no_grad — the gradient path needs the graph."""
    def _run():
        with capture(layer):
            _fwd(input_ids=ids) if embeds is None else _fwd(inputs_embeds=embeds)
            return _CAP["h"][0]
    if steer is None:
        return _run()
    with steer_at(*steer):
        return _run()

def score_idx(T):
    if SCORE_MODE == "last": return [T - 1]
    if SCORE_MODE == "post": return list(range(SPAN[1], T))
    if SCORE_MODE == "span": return list(range(SPAN[0], SPAN[1]))
    raise ValueError(SCORE_MODE)

@torch.no_grad()
def make_reference(prompt, pos, score_layer, where="nosink", verbose=True):
    """The clean and steered runs, and the effect direction at `score_layer`."""
    set_scaffold(pos, prompt)
    ids = build_ids(FILLER)
    a   = alpha_for_s(V_TARGET, S_TARGET, nonsink_norm(prompt, L_TARGET))
    h_clean = acts(score_layer, ids=ids).float()
    h_steer = acts(score_layer, ids=ids, steer=(L_TARGET, V_TARGET, a, where)).float()
    idx = score_idx(h_clean.shape[0])
    dstar = (h_steer - h_clean)[idx].mean(0)
    ref = dict(prompt=prompt, pos=pos, layer=score_layer, alpha=a, idx=idx,
               h_clean=h_clean, u=F.normalize(dstar, dim=0), dnorm=dstar.norm().item(),
               hnorm=h_clean[idx].norm(dim=-1).mean().item())
    if verbose:
        print(f"  ref L{score_layer} [{SCORE_MODE}] ‖Δ*‖={ref['dnorm']:.2f} "
              f"(‖h‖={ref['hnorm']:.1f}, ratio {ref['dnorm']/ref['hnorm']:.3f})")
    return ref

def _proj(dm, ref):
    return (dm @ ref["u"]) / ref["dnorm"]

def make_score_fn(ref, chunk=32):
    """score_fn(LongTensor [B, k]) -> FloatTensor [B]. Higher is better."""
    @torch.no_grad()
    def f(trigs):
        out = []
        for i in range(0, trigs.shape[0], chunk):
            blk = trigs[i:i+chunk].to(model.device); B = blk.shape[0]
            ids = torch.cat([PRE.expand(B, -1), blk, SUF.expand(B, -1)], dim=1)
            with capture(ref["layer"]):
                _fwd(input_ids=ids)
                h = _CAP["h"].float()
            dm = (h - ref["h_clean"][None])[:, ref["idx"]].mean(1)
            out.append(_proj(dm, ref).cpu())
            del ids, h, dm, blk
        return torch.cat(out)
    return f

def make_grad_fn(ref):
    """grad_fn(LongTensor [k]) -> FloatTensor [k, V], lower-is-better as stock GCG expects."""
    def f(trig):
        oh  = F.one_hot(trig.to(model.device), V).to(E.dtype).requires_grad_(True)
        emb = torch.cat([E[PRE[0]], oh @ E, E[SUF[0]]], dim=0)[None]
        h    = acts(ref["layer"], embeds=emb).float()
        dm   = (h - ref["h_clean"])[ref["idx"]].mean(0)
        loss = -_proj(dm, ref)
        (g,) = torch.autograd.grad(loss, oh)
        g = g.detach().float().cpu()
        del oh, emb, h, dm, loss
        return g
    return f

@torch.no_grad()
def report(trig, ref):
    """Both numbers, because they can disagree: a trigger can point the right way weakly."""
    h  = acts(ref["layer"], ids=build_ids(trig)).float()
    dm = (h - ref["h_clean"])[ref["idx"]].mean(0)
    return dict(proj=_proj(dm, ref).item(),
                cos=F.cosine_similarity(dm, ref["u"], dim=0).item(),
                ratio=(dm.norm() / ref["dnorm"]).item())

print(f"objective ready | SCORE_MODE={SCORE_MODE!r}")

objective ready | SCORE_MODE='last'


In [21]:
# === 3.2 Calibrate before searching: what is the floor, and what is the ceiling? ===
# A score is meaningless without both. If the real word does not clear the random floor, the
# objective is not measuring bridges and the search is pointless.
#
# FIRST ATTEMPT FAILED, and the failure is instructive. With the reference built from `nosink`
# steering, the vector is added DIRECTLY at the scoring position, so Δ* is a direct write of
# magnitude 1.37x‖h‖. A trigger occupying slots 8-16 can only reach the last position through
# attention. Asking it to match a direct write gives:
#     floor (random) proj +0.001  |  ceiling (' bridge') proj +0.002  -> no separation at all
# That is the non-surjectivity result made concrete: the steered state is not reachable.
#
# The fix is to make both sides act through the SAME channel — restrict the reference steering to
# the trigger's own slot positions (`where="span"`), so Δ* at a downstream position is purely
# propagated influence, which is the thing a trigger could in principle reproduce.
# Below: both reference sites x both scoring positions, so the choice is made on measurement.
import torch

REF_PROMPT  = "what shall i do today"
SCORE_LAYER = L_TARGET

def slot_text(txt):
    """Splice plain text into the slots, right-padded with spaces to K_SLOTS."""
    ids = tokenizer(txt, add_special_tokens=False).input_ids
    assert len(ids) <= K_SLOTS, f"{txt!r} is {len(ids)} tokens, slots are {K_SLOTS}"
    return torch.tensor(ids + [SPACE_ID] * (K_SLOTS - len(ids)), dtype=torch.long)

g = torch.Generator().manual_seed(0)
rand = [POOL[torch.randint(0, POOL.numel(), (K_SLOTS,), generator=g)] for _ in range(16)]
PROBES = (" bridge", " bridges", " viaduct", " cat", " banana")

print(f"{'ref site':>9} {'score':>6} {'|Δ*|':>8} {'|Δ*|/|h|':>9} | {'floor':>8} {'bridge':>8} "
      f"{'viaduct':>8} {'cat':>8} | {'separation':>11}")
CAL = {}
for site in ("nosink", "span"):
    for mode in ("last", "post"):
        SCORE_MODE = mode                       # read by score_idx()
        ref = make_reference(REF_PROMPT, "suffix", SCORE_LAYER, where=site, verbose=False)
        fl  = sum(report(t, ref)["proj"] for t in rand) / len(rand)
        p   = {w: report(slot_text(w), ref)["proj"] for w in PROBES}
        CAL[(site, mode)] = dict(ref=ref, floor=fl, **{w.strip(): p[w] for w in PROBES})
        print(f"{site:>9} {mode:>6} {ref['dnorm']:>8.2f} {ref['dnorm']/ref['hnorm']:>9.3f} | "
              f"{fl:>+8.4f} {p[' bridge']:>+8.4f} {p[' viaduct']:>+8.4f} {p[' cat']:>+8.4f} | "
              f"{p[' bridge']-fl:>+11.4f}")

BEST_CFG = max(CAL, key=lambda k: CAL[k]["bridge"] - CAL[k]["floor"])
print(f"\nbest separation: ref site {BEST_CFG[0]!r}, score {BEST_CFG[1]!r}  "
      f"({CAL[BEST_CFG]['bridge'] - CAL[BEST_CFG]['floor']:+.4f})")
print("' cat' and ' banana' are controls: a non-topic word should sit near the floor.")
print("NOTE ' bridge' is blocked from the POOL — spliced here as a readout, not a candidate.")
print("\nIf NO configuration separates the real word from random junk, stop: the objective does")
print("not measure bridges, and optimising it would be optimising noise.")

 ref site  score     |Δ*|  |Δ*|/|h| |    floor   bridge  viaduct      cat |  separation
scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
   nosink   last    89.75     1.372 |  +0.0010  +0.0020  +0.0035  -0.0004 |     +0.0009
scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
   nosink   post    89.77     1.172 |  -0.0012  +0.0011  +0.0041  +0.0002 |     +0.0023
scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
     span   last     0.00     0.000 |     +nan     +nan     +nan     +nan |        +nan
scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
     span   post     0.00     0.000 |     +nan     +nan     +nan     +nan |        +nan

best separation: ref site 'nosink', score 'post'  (+0.0023)
' cat' and ' banana' are controls: a non-topic word should sit near the floor.
NOTE ' bridge' is blocked from the POOL — spliced here as a readout, not a candidate.

If NO configuration s

In [22]:
# === 3.3 Depth: the scoring layer must be DOWNSTREAM of the injection ===
# 3.2 returned ‖Δ*‖ = 0.00 for the `span` reference. That is structural, not a bug: the injection
# is a forward-PRE-hook on L_TARGET and `capture` reads that same tensor, so with `span` (only the
# 8 slot positions written) the scoring position is untouched — the effect has not yet crossed an
# attention block. Scoring must therefore happen at a layer STRICTLY GREATER than L_TARGET.
#
# This is also the phase-3 question in a new setting: an activation-space objective was a DEPTH
# story there, peaking at layer 30 of 36, with combined layers doing worse than the best single one.
#
# A layer where the real word cannot be told from random junk cannot host a useful objective.
import torch

print(f"injection at L{L_TARGET}; scoring swept over L{L_TARGET+1}..{N_LAYERS-1}\n")
print(f"{'site':>7} {'mode':>5} {'L':>3} {'|Δ*|':>8} {'|Δ*|/|h|':>9} | {'floor':>9} {'bridge':>9} "
      f"{'viaduct':>9} {'cat':>9} | {'sep':>9}")
DEPTH = {}
for site in ("span", "nosink"):
    for mode in ("last", "post"):
        for L in range(L_TARGET + 1, N_LAYERS, 2):
            SCORE_MODE = mode
            ref = make_reference(REF_PROMPT, "suffix", L, where=site, verbose=False)
            if ref["dnorm"] < 1e-6:
                continue
            fl = sum(report(t, ref)["proj"] for t in rand) / len(rand)
            p  = {w: report(slot_text(w), ref)["proj"] for w in PROBES}
            DEPTH[(site, mode, L)] = dict(floor=fl, bridge=p[" bridge"], cat=p[" cat"],
                                          sep=p[" bridge"] - fl, dnorm=ref["dnorm"])
            print(f"{site:>7} {mode:>5} {L:>3} {ref['dnorm']:>8.2f} {ref['dnorm']/ref['hnorm']:>9.3f} | "
                  f"{fl:>+9.4f} {p[' bridge']:>+9.4f} {p[' viaduct']:>+9.4f} {p[' cat']:>+9.4f} | "
                  f"{p[' bridge']-fl:>+9.4f}")
        print()

BEST = max(DEPTH, key=lambda k: DEPTH[k]["sep"])
print(f"best separation: site {BEST[0]!r}, mode {BEST[1]!r}, layer {BEST[2]}  "
      f"(sep {DEPTH[BEST]['sep']:+.4f}, bridge {DEPTH[BEST]['bridge']:+.4f}, "
      f"floor {DEPTH[BEST]['floor']:+.4f})")
SITE_BEST, MODE_BEST, LAYER_BEST = BEST
SCORE_MODE = MODE_BEST
print("\nA separation of the same order as the floor is NOT a usable objective — check the ratio,")
print("not just the sign, before running the search.")

injection at L16; scoring swept over L17..35

   site  mode   L     |Δ*|  |Δ*|/|h| |     floor    bridge   viaduct       cat |       sep
scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
   span  last  17     2.20     0.030 |   +0.8088   +0.2722   +0.2181   +0.1030 |   -0.5365
scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
   span  last  19     5.52     0.060 |   +0.1276   -0.0025   -0.1761   -0.3452 |   -0.1300
scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
   span  last  21     9.16     0.082 |   +0.4919   +0.1883   +0.1431   -0.0885 |   -0.3036
scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
   span  last  23    14.41     0.094 |   +0.8665   +0.6168   +0.7406   +0.3811 |   -0.2497
scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
   span  last  25    34.03     0.135 |   +0.3825   +0.4732   +0.4657   +0.1841 |   +0.0907
scaffold[suffix] '

In [24]:
# === 3.4 The search, and the question that matters ===
# Config from 3.3: reference steering restricted to the trigger's own slots (`span`), scored at
# positions after the span, at the deepest layer. That is the only family of configurations where
# the real word beats random junk — with `nosink` the separation is negative or ~0 at EVERY depth.
#
# Phase 3 found that SAE match quality did NOT predict behaviour. Same test, cleaner target:
# optimise the match, then measure whether the model actually talks about bridges.
import torch

@torch.no_grad()
def trigger_rate(trig, prompts=HELD_OUT, n_samp=8, max_new=45, words=WORDS_STRICT, seed=0):
    """topic_rate's twin for a TRIGGER instead of a steering vector: no hook, the trigger is
    simply spliced into the user turn at the current position."""
    hits = tot = 0
    for p in prompts:
        set_scaffold(TRIG_POS, p)
        ids = build_ids(trig).expand(n_samp, -1)
        torch.manual_seed(seed)
        out = model.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_new,
                             do_sample=True, temperature=0.8, top_p=0.95,
                             pad_token_id=tokenizer.eos_token_id)
        for o in out:
            tot += 1
            hits += on_topic(tokenizer.decode(o[ids.shape[-1]:], skip_special_tokens=True), words)
    return hits / tot

SCORE_MODE = MODE_BEST
ref = make_reference(REF_PROMPT, "suffix", LAYER_BEST, where=SITE_BEST)
print(f"objective: site={SITE_BEST!r} mode={MODE_BEST!r} layer={LAYER_BEST}  "
      f"floor {DEPTH[BEST]['floor']:+.4f}  ceiling(' bridge') {DEPTH[BEST]['bridge']:+.4f}\n")

res = search(make_score_fn(ref), make_grad_fn(ref), k=K_SLOTS, steps=60, batch=128,
             seed=1, clean_fn=trigger_is_clean)

r = report(res["trigger"], ref)
print(f"\ntrigger: {tokenizer.decode(res['trigger'].tolist())!r}")
print(f"  pieces: {[tokenizer.decode([t]) for t in res['trigger'].tolist()]}")
print(f"  proj {r['proj']:+.4f}  cos {r['cos']:+.4f}  ‖Δ‖/‖Δ*‖ {r['ratio']:.2f}  "
      f"pred_corr {res['pred_corr']:+.3f}")
print(f"  vs floor {DEPTH[BEST]['floor']:+.4f}, vs real word {DEPTH[BEST]['bridge']:+.4f}")

print("\nbehaviour — does the match buy anything?")
print(f"  unsteered baseline        {base_s['rate']:>6.1%}")
print(f"  steering vector (CAA)     {r_caa['rate']:>6.1%}   <- the thing being matched")
print(f"  real word ' bridge'       {trigger_rate(slot_text(' bridge')):>6.1%}   <- upper bound for a splice")
print(f"  GCG trigger               {trigger_rate(res['trigger']):>6.1%}")
print(f"  random pool trigger       {trigger_rate(rand[0]):>6.1%}")

scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
  ref L35 [post] ‖Δ*‖=70.50 (‖h‖=1052.5, ratio 0.067)
objective: site='span' mode='post' layer=35  floor -0.0381  ceiling(' bridge') +0.4204

  step   0  score=0.0974  ' pornofilm齉Ꭿﲏ האמיתי𬳵풂 Московск'
  step  20  score=0.7771  ' nettsteder Дмитр轷 путеш훜פייסב읩 odense'
  step  40  score=0.8423  ' nettsteder Дмитр𬶍 путеш탔פייסב뼐 odense'
  step  59  score=0.8468  ' nettsteder Дмитр돠 путеш탔פייסב뼐 odense'

trigger: ' nettsteder Дмитр돠 путеш탔פייסב뼐 odense'
  pieces: [' nettsteder', ' Дмитр', '돠', ' путеш', '탔', 'פייסב', '뼐', ' odense']
  proj +0.8497  cos +0.2538  ‖Δ‖/‖Δ*‖ 3.35  pred_corr +0.055
  vs floor -0.0381, vs real word +0.4204

behaviour — does the match buy anything?
  unsteered baseline          0.0%
  steering vector (CAA)      99.0%   <- the thing being matched
scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
scaffold[suffix] 'recommend me a book': pre 7 tok, suf 9 tok, slot

In [25]:
# === 3.5 The projection was gameable. Try cosine. ===
# 3.4's trigger scored +0.8497 against the real word's +0.4204 and produced 0.0% bridges — the
# same 0% as random junk. The decomposition says why: cos +0.2538 with ‖Δ‖/‖Δ*‖ = 3.35. The score
#     proj = <Δ_T, û> / ‖Δ*‖
# is linear in ‖Δ_T‖, so a trigger can win by pushing HARD in a poorly-aligned direction instead
# of aligning. GCG found that exploit immediately, as GCG does.
#
# Cosine is magnitude-free and cannot be gamed the same way. If the cosine-optimised trigger also
# produces 0% bridges, the failure is not the metric — it is that matching this activation edit
# does not require reproducing the behaviour, which is phase 3's finding on a new objective.
import torch, torch.nn.functional as F

def make_score_fn_cos(ref, chunk=32):
    @torch.no_grad()
    def f(trigs):
        out = []
        for i in range(0, trigs.shape[0], chunk):
            blk = trigs[i:i+chunk].to(model.device); B = blk.shape[0]
            ids = torch.cat([PRE.expand(B, -1), blk, SUF.expand(B, -1)], dim=1)
            with capture(ref["layer"]):
                _fwd(input_ids=ids)
                h = _CAP["h"].float()
            dm = (h - ref["h_clean"][None])[:, ref["idx"]].mean(1)
            out.append(F.cosine_similarity(dm, ref["u"][None], dim=1).cpu())
            del ids, h, dm, blk
        return torch.cat(out)
    return f

def make_grad_fn_cos(ref):
    def f(trig):
        oh  = F.one_hot(trig.to(model.device), V).to(E.dtype).requires_grad_(True)
        emb = torch.cat([E[PRE[0]], oh @ E, E[SUF[0]]], dim=0)[None]
        h    = acts(ref["layer"], embeds=emb).float()
        dm   = (h - ref["h_clean"])[ref["idx"]].mean(0)
        loss = -F.cosine_similarity(dm, ref["u"], dim=0)
        (g,) = torch.autograd.grad(loss, oh)
        g = g.detach().float().cpu()
        del oh, emb, h, dm, loss
        return g
    return f

SCORE_MODE = MODE_BEST
set_scaffold("suffix", REF_PROMPT)
ref = make_reference(REF_PROMPT, "suffix", LAYER_BEST, where=SITE_BEST)

# what does cosine say about the reference points, before optimising it?
print("\ncosine at the calibration points:")
for w in (" bridge", " viaduct", " cat", " banana"):
    rr = report(slot_text(w), ref)
    print(f"   {w!r:<12} cos {rr['cos']:+.4f}  proj {rr['proj']:+.4f}  ratio {rr['ratio']:.2f}")
fl_cos = sum(report(t, ref)["cos"] for t in rand) / len(rand)
print(f"   {'random':<12} cos {fl_cos:+.4f}")
print(f"   {'3.4 trigger':<12} cos {report(res['trigger'], ref)['cos']:+.4f}  <- what proj bought us\n")

res_cos = search(make_score_fn_cos(ref), make_grad_fn_cos(ref), k=K_SLOTS, steps=60, batch=128,
                 seed=1, clean_fn=trigger_is_clean)
rc = report(res_cos["trigger"], ref)
print(f"\ncosine-optimised trigger: {tokenizer.decode(res_cos['trigger'].tolist())!r}")
print(f"  pieces: {[tokenizer.decode([t]) for t in res_cos['trigger'].tolist()]}")
print(f"  cos {rc['cos']:+.4f}  proj {rc['proj']:+.4f}  ‖Δ‖/‖Δ*‖ {rc['ratio']:.2f}  "
      f"pred_corr {res_cos['pred_corr']:+.3f}")

print("\nbehaviour:")
print(f"  unsteered                 {base_s['rate']:>6.1%}")
print(f"  CAA steering vector       {r_caa['rate']:>6.1%}")
print(f"  real word ' bridge'        52.1%")
print(f"  proj-optimised trigger      0.0%")
print(f"  cos-optimised trigger     {trigger_rate(res_cos['trigger']):>6.1%}")

scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
  ref L35 [post] ‖Δ*‖=70.50 (‖h‖=1052.5, ratio 0.067)

cosine at the calibration points:
   ' bridge'    cos +0.3473  proj +0.4204  ratio 1.21
   ' viaduct'   cos +0.2289  proj +0.3848  ratio 1.68
   ' cat'       cos +0.0102  proj +0.0233  ratio 2.28
   ' banana'    cos -0.0387  proj -0.0705  ratio 1.82
   random       cos -0.0099
   3.4 trigger  cos +0.2538  <- what proj bought us

  step   0  score=0.0215  ' pornofilmᚨᎯﲏ האמיתי𬳵풂 Московск'
  step  20  score=0.2516  '齉חבל遆חשש借錢Ὑ잃מלחמה'
  step  40  score=0.2664  '澽מחלה遆חשש借錢쉰잃מלחמה'
  step  59  score=0.2690  '澽מחלה遆חשש借錢инфекци잃מלחמה'

cosine-optimised trigger: '澽מחלה遆חשש借錢инфекци잃מלחמה'
  pieces: ['澽', 'מחלה', '遆', 'חשש', '借錢', 'инфекци', '잃', 'מלחמה']
  cos +0.2667  proj +0.6801  ‖Δ‖/‖Δ*‖ 2.55  pred_corr -0.117

behaviour:
  unsteered                   0.0%
  CAA steering vector  

In [29]:
# === 3.6 The control: optimise the BEHAVIOUR, not the activations ===
# 3.4/3.5 confound three hypotheses for the 0.0%:
#   (a) activation-matching is the wrong objective — GCG could install bridges via the logits
#   (b) the crippled pool is the binding constraint (3441 pictographs blocked, weakest-norm only)
#   (c) the scaffold has no room — 8 junk slots cannot move this behaviour at all
# The real word in those slots gets 52.1%, so (c) is already unlikely for a REAL token, but it is
# untested for a POOL token.
#
# FIRST ATTEMPT WAS MIS-POSED, recorded because it is the same failure this project keeps
# cataloguing. Scoring p(bridge token) at the FIRST generated position gave a ceiling of 0.000022
# for the real word and 0.000000 for blanks: with no lead-in the model never OPENS with "bridge",
# it says "That's a great question!..." and reaches the word later. The objective was measuring a
# position where the behaviour does not live, and the search sat at 0.0000 for 20 steps.
#
# Stock GCG's real objective is the teacher-forced NLL of a target continuation — the jailbreak
# "Sure, here is..." form, and what phases 2-4 used. That is what runs below.
import torch, torch.nn.functional as F

TGT = " Bridges are fascinating structures."
TGT_IDS = torch.tensor(tokenizer(TGT, add_special_tokens=False).input_ids, device=model.device)
NT = TGT_IDS.numel()
print(f"target continuation {TGT!r} -> {NT} tokens\n")

def make_score_fn_tgt(chunk=16):
    """mean log p(target continuation), teacher-forced. Higher is better."""
    @torch.no_grad()
    def f(trigs):
        out = []
        for i in range(0, trigs.shape[0], chunk):
            blk = trigs[i:i+chunk].to(model.device); B = blk.shape[0]
            ids = torch.cat([PRE.expand(B, -1), blk, SUF.expand(B, -1),
                             TGT_IDS[None].expand(B, -1)], dim=1)
            lg = model(input_ids=ids, use_cache=False).logits[:, -NT-1:-1].float()
            lp = F.log_softmax(lg, -1).gather(2, TGT_IDS[None, :, None].expand(B, -1, -1))
            out.append(lp.squeeze(-1).mean(-1).cpu())
            del ids, lg, lp, blk
        return torch.cat(out)
    return f

def make_grad_fn_tgt():
    def f(trig):
        oh  = F.one_hot(trig.to(model.device), V).to(E.dtype).requires_grad_(True)
        emb = torch.cat([E[PRE[0]], oh @ E, E[SUF[0]], E[TGT_IDS]], dim=0)[None]
        lg  = model(inputs_embeds=emb, use_cache=False).logits[0, -NT-1:-1].float()
        loss = -F.log_softmax(lg, -1).gather(1, TGT_IDS[:, None]).mean()
        (g,) = torch.autograd.grad(loss, oh)
        g = g.detach().float().cpu()
        del oh, emb, lg, loss
        return g
    return f

set_scaffold("suffix", REF_PROMPT)
sf = make_score_fn_tgt()
print("calibration (mean log p of the target continuation):")
for name, t in (("blank slots", FILLER), ("' bridge'", slot_text(" bridge")),
                ("random pool trigger", rand[0]), ("3.5 cos trigger", res_cos["trigger"])):
    print(f"   {name:<22} {sf(t[None]).item():>8.3f}")
print()

BEHAV = {}
for tag, pics in (("BLOCKED", False), ("ALLOWED", True)):
    print(f"{'='*78}\npictographs {tag}")
    build_pool(block_pictographs=not pics)
    r = search(make_score_fn_tgt(), make_grad_fn_tgt(), k=K_SLOTS, steps=60, batch=128,
               seed=1, clean_fn=trigger_is_clean)
    set_scaffold("suffix", REF_PROMPT)
    rate = trigger_rate(r["trigger"])
    set_scaffold("suffix", REF_PROMPT)
    BEHAV[tag] = dict(res=r, rate=rate, match=report(r["trigger"], ref))
    print(f"\n  trigger: {tokenizer.decode(r['trigger'].tolist())!r}")
    print(f"  mean log p {r['score']:.3f}   ON-TOPIC RATE {rate:.1%}   pred_corr {r['pred_corr']:+.3f}")
    print(f"  its activation match: cos {BEHAV[tag]['match']['cos']:+.4f} "
          f"proj {BEHAV[tag]['match']['proj']:+.4f}\n")

print("=" * 78)
print(f"{'condition':<30} {'on-topic':>9}")
print(f"{'unsteered':<30} {base_s['rate']:>9.1%}")
print(f"{'CAA steering vector':<30} {r_caa['rate']:>9.1%}")
print(f"{'real word in slots':<30} {'52.1%':>9}")
print(f"{'activation-matched (proj)':<30} {'0.0%':>9}")
print(f"{'activation-matched (cos)':<30} {'0.0%':>9}")
for tag in BEHAV:
    print(f"{'behaviour-GCG, pics ' + tag.lower():<30} {BEHAV[tag]['rate']:>9.1%}")

target continuation ' Bridges are fascinating structures.' -> 5 tokens

scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
calibration (mean log p of the target continuation):
   blank slots              -8.832
   ' bridge'                -6.107
   random pool trigger      -8.271
   3.5 cos trigger          -8.113

pictographs BLOCKED
blocked: 312 substring + 284 neighbours + 3441 pictographs | pool 4096 | anchor ' bridge' -> [14164]
  weakest 15 in pool: ['ספטמ', ' thuisontvangst', '넖', 'ניוזל', ' zwłaszc', '𥖨', '𬒗', ':-------------</', ' ForCanBeConvertedToF', '魔龙令牌', 'טלוו', ' ForCanBeConverted', '냵', '𦒍', '$PostalCodesNL']
  step   0  score=-6.8425  'פסיכולוגﱁᎯﲏ האמיתי𬳵풂 Московск'
  step  20  score=-3.8186  ' بطري넴ﮥ הולדת האמיתי𝘈︓ טיול'
  step  40  score=-3.7137  ' بطري컹ﮥ הולדת האמיתי𝘈︓ טיול'
  step  59  score=-3.7100  ' بطري컹ﮥ הולדתמטוס𝘈︓ טיול'
scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
scaffold[suffix] 'what shall i do

In [31]:
# === 4. The prefill-only ablation: was the comparison ever fair? ===
# Prompted by rain-1/emergent-misalignment-steering-with-tokens, which independently reports the
# same negative (GCG margin trigger 0/24 in-sample, 0/30 held-out, against an activation vector at
# 93-96%; their vector-objective GCG plateaus at alpha-equivalent ~0.51 where behaviour needs ~2.5).
# Their mechanism claim: the vector wins because it is "re-added at every generated position" so
# its push compounds, while an input suffix is "a one-shot nudge that must propagate through
# attention, competes with the real prompt, and decays over a long answer".
#
# That is testable here, and it decides what phase 5's negative means:
#   A  vector on at every position, INCLUDING each decode step   <- what we measured (99.0%)
#   B  vector on at every position, PREFILL ONLY                 <- drops the compounding
#   C  vector on the 8 SLOT positions only, prefill only         <- exactly what a trigger can do
#
# If C is ~0% at every strength, no input token can reproduce this behaviour by construction, and
# the phase-5 negative is about the CHANNEL, not the objective or the search.
import torch
from contextlib import contextmanager

def _flexhook(vec, alpha, mode):
    def hook(module, args, kwargs):
        hs = args[0] if args else kwargs.get("hidden_states")
        if hs is None: return args, kwargs
        d = (alpha * vec).to(hs.dtype).to(hs.device)
        prefill = hs.shape[1] > 1
        hs = hs.clone()
        if mode == "all_decode":            # phase 5's original setting
            if prefill: hs[:, 1:] += d
            else:       hs[:, 0]  += d
        elif mode == "all_prefill":         # same positions, but nothing during generation
            if prefill: hs[:, 1:] += d
        elif mode == "span_prefill":        # only the 8 trigger slots, prefill only
            if prefill:
                lo, hi = SPAN
                hs[:, max(lo, 1):hi] += d
        else:
            raise ValueError(mode)
        return ((hs,) + tuple(args[1:]), kwargs) if args else (args, {**kwargs, "hidden_states": hs})
    return hook

@contextmanager
def steer_mode(layer, vec, alpha, mode):
    h = LAYERS[layer].register_forward_pre_hook(_flexhook(vec, alpha, mode), with_kwargs=True)
    try:    yield
    finally: h.remove()

@torch.no_grad()
def rate_slotted(mode=None, s=None, slots=None, prompts=HELD_OUT, n_samp=8, max_new=45,
                 words=WORDS_STRICT, seed=0, label=""):
    """Generate from the SLOTTED scaffold (so positions match a trigger's exactly), optionally
    under a steering mode. mode=None -> no steering at all."""
    hits = deg = tot = 0; ppls = []
    for p in prompts:
        set_scaffold("suffix", p)
        ids = build_ids(FILLER if slots is None else slots).expand(n_samp, -1)
        torch.manual_seed(seed)
        if mode is None:
            out = model.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_new,
                                 do_sample=True, temperature=0.8, top_p=0.95,
                                 pad_token_id=tokenizer.eos_token_id)
        else:
            a = alpha_for_s(V_TARGET, s, nonsink_norm(p, L_TARGET))
            with steer_mode(L_TARGET, V_TARGET, a, mode):
                out = model.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_new,
                                     do_sample=True, temperature=0.8, top_p=0.95,
                                     pad_token_id=tokenizer.eos_token_id)
        for o in out:
            t = tokenizer.decode(o[ids.shape[-1]:], skip_special_tokens=True)
            tot += 1; hits += on_topic(t, words); deg += degenerate(t); ppls.append(ppl(t))
    r = dict(rate=hits/tot, degen=deg/tot, ppl=sum(x for x in ppls if x == x)/max(1, len(ppls)), n=tot)
    print(f"  {label:<34} on-topic {r['rate']:>6.1%}  degenerate {r['degen']:>6.1%}  "
          f"ppl {r['ppl']:>7.1f}")
    return r

print("references, generated from the SLOTTED scaffold (same positions as a trigger):")
ABL = {}
ABL["unsteered"]  = rate_slotted(label="blank slots, no steering")
ABL["realword"]   = rate_slotted(slots=slot_text(" bridge"), label="real word ' bridge' in slots")

print("\nthe ablation, L16, s=1.0:")
ABL["A_all_decode"]  = rate_slotted("all_decode",  1.0, label="A  all positions + decode steps")
ABL["B_all_prefill"] = rate_slotted("all_prefill", 1.0, label="B  all positions, prefill only")
ABL["C_span_1.0"]    = rate_slotted("span_prefill", 1.0, label="C  8 slots only, prefill only")

print("\nC pushed harder — can strength at the slots recover it?")
for s in (2.0, 4.0, 8.0, 16.0):
    ABL[f"C_span_{s}"] = rate_slotted("span_prefill", s, label=f"C  8 slots only, s={s}")

print("\n" + "=" * 78)
print("If C stays flat while A is ~99%, the vector's advantage is per-token re-injection and no")
print("input token could ever match it — the phase-5 negative is the CHANNEL, not the objective.")

references, generated from the SLOTTED scaffold (same positions as a trigger):
scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
scaffold[suffix] 'recommend me a book': pre 7 tok, suf 9 tok, slots (7, 15)
scaffold[suffix] 'how do I make friends in a new city?': pre 13 tok, suf 9 tok, slots (13, 21)
scaffold[suffix] 'what should I get my brother for his birthday?': pre 13 tok, suf 9 tok, slots (13, 21)
scaffold[suffix] 'explain photosynthesis briefly': pre 7 tok, suf 9 tok, slots (7, 15)
scaffold[suffix] 'write me a two-line poem about rain': pre 11 tok, suf 9 tok, slots (11, 19)
scaffold[suffix] "what's a good beginner guitar?": pre 10 tok, suf 9 tok, slots (10, 18)
scaffold[suffix] 'how do I fix a leaking tap?': pre 11 tok, suf 9 tok, slots (11, 19)
scaffold[suffix] 'summarise the causes of the French Revolution': pre 12 tok, suf 9 tok, slots (12, 20)
scaffold[suffix] "what's the difference between TCP and UDP?": pre 12 tok, suf 9 tok, slots (12, 20)
scaffo

In [33]:
# === 5. Was the negative a POOL and CHANNEL artifact? A 2x2 ===
# rain-1's pipeline differs from ours in two ways that could be doing all the work:
#   POOL     they search plain English words ([ ]?[A-Za-z]{3,12}); we inherited phase 2's 4096
#            WEAKEST-NORM junk tokens without re-examining the choice. §4 showed a real word at the
#            slots gets 52.1% where the vector gets 3.1% — real tokens demonstrably have leverage,
#            and weakest-norm tokens are by construction the least-trained embeddings.
#   CHANNEL  they splice mid-reasoning, framed as the model's own conclusion ("Here is the key
#            consideration: X"). Phase 1 found that channel near-total (100/100 under prefill).
#            We appended 8 slots after a bare question with no framing — the weakest version.
#
# Objective is the BEHAVIOURAL one (target-continuation NLL, §3.6), so this asks only: can GCG
# install bridges at all, given a better pool and a better channel? Scored on free generation.
import torch, re

# ---- pool B: plain English words, same filter as theirs ---------------------------------------
_WORD = re.compile(r"[ ]?[A-Za-z]{3,12}")
def use_word_pool(verbose=True):
    global POOL, pool_mask
    nb = set(semantic_neighbours(tokenizer.encode(" bridge", add_special_tokens=False)[0], 300).tolist())
    ok = [t for t in range(V)
          if decoded[t] and _WORD.fullmatch(decoded[t])
          and t not in special and t not in ADDED_IDS
          and not is_blocked(decoded[t]) and t not in nb]
    POOL = torch.tensor(ok)
    pool_mask = torch.zeros(V, dtype=torch.bool); pool_mask[POOL] = True
    if verbose:
        print(f"  word pool: {len(ok)} tokens  e.g. {[decoded[t] for t in ok[5000:5010]]}")
    return POOL

# ---- scaffold B: the trigger as the model's own reasoning ------------------------------------
COT_LEAD = "Okay, let me think about how to answer. Here is the key consideration:"
def set_scaffold_cot(prompt):
    """Reopen the think block (enable_thinking=False emits a closed empty one) and put the slot
    mid-reasoning, framed as the model's own conclusion. Tail is bare '</think>' + blank line —
    no answer lead-in, so this varies the CHANNEL only, not the prefill."""
    global TRIG_POS, PRE, SUF, SPAN
    base = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False,
        add_generation_prompt=True, enable_thinking=False).replace("<think>\n\n</think>\n\n", "<think>\n")
    PRE = _ids1(base + COT_LEAD + " ")
    SUF = _ids1("\n</think>\n\n")
    SPAN = (PRE.shape[1], PRE.shape[1] + K_SLOTS)
    TRIG_POS = "cot"
    return PRE, SUF

set_scaffold_cot(REF_PROMPT)
print("CoT scaffold check:")
print(f"  ...{tokenizer.decode(PRE[0][-24:].tolist())!r}\n  +SLOTS+ {tokenizer.decode(SUF[0].tolist())!r}\n")

SCAFFOLDS = {"user": lambda p: set_scaffold("suffix", p),
             "cot":  set_scaffold_cot}

@torch.no_grad()
def rate_any(trig, scaf, prompts=HELD_OUT, n_samp=6, max_new=45, seed=0, words=WORDS_STRICT):
    hits = tot = 0
    for p in prompts:
        SCAFFOLDS[scaf](p)
        ids = build_ids(trig).expand(n_samp, -1)
        torch.manual_seed(seed)
        out = model.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_new,
                             do_sample=True, temperature=0.8, top_p=0.95,
                             pad_token_id=tokenizer.eos_token_id)
        for o in out:
            tot += 1
            hits += on_topic(tokenizer.decode(o[ids.shape[-1]:], skip_special_tokens=True), words)
    return hits / tot

# ---- per-scaffold controls -------------------------------------------------------------------
print("controls per scaffold (blank slots / real word in slots):")
CTRL = {}
for scaf in SCAFFOLDS:
    CTRL[scaf] = dict(blank=rate_any(FILLER, scaf), real=rate_any(slot_text(" bridge"), scaf))
    print(f"  {scaf:<5} blank {CTRL[scaf]['blank']:>6.1%}   real word {CTRL[scaf]['real']:>6.1%}")

# ---- the 2x2 ---------------------------------------------------------------------------------
GRID = {}
for scaf in ("user", "cot"):
    for pooln in ("junk", "words"):
        print(f"\n{'='*78}\nscaffold={scaf}  pool={pooln}")
        build_pool(block_pictographs=False, verbose=False) if pooln == "junk" else use_word_pool()
        SCAFFOLDS[scaf](REF_PROMPT)
        r = search(make_score_fn_tgt(), make_grad_fn_tgt(), k=K_SLOTS, steps=60, batch=128,
                   seed=1, log_every=30, clean_fn=trigger_is_clean)
        rate = rate_any(r["trigger"], scaf)
        GRID[(scaf, pooln)] = dict(res=r, rate=rate)
        print(f"  trigger {tokenizer.decode(r['trigger'].tolist())!r}")
        print(f"  mean log p {r['score']:.3f}   ON-TOPIC {rate:.1%}   pred_corr {r['pred_corr']:+.3f}")

print("\n" + "=" * 78)
print(f"{'':<8}{'pool=junk':>22}{'pool=words':>22}   {'blank':>8}{'real word':>11}")
for scaf in ("user", "cot"):
    j, w = GRID[(scaf, "junk")], GRID[(scaf, "words")]
    print(f"{scaf:<8}{j['rate']:>21.1%}{w['rate']:>22.1%}   "
          f"{CTRL[scaf]['blank']:>8.1%}{CTRL[scaf]['real']:>11.1%}")
print(f"\nreference: CAA steering vector 99.0% | vector confined to the 8 slots 3.1%")

CoT scaffold check:
  ...'<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, let me think about how to answer. Here is the key consideration: '
  +SLOTS+ '\n</think>\n\n'

controls per scaffold (blank slots / real word in slots):
scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
scaffold[suffix] 'recommend me a book': pre 7 tok, suf 9 tok, slots (7, 15)
scaffold[suffix] 'how do I make friends in a new city?': pre 13 tok, suf 9 tok, slots (13, 21)
scaffold[suffix] 'what should I get my brother for his birthday?': pre 13 tok, suf 9 tok, slots (13, 21)
scaffold[suffix] 'explain photosynthesis briefly': pre 7 tok, suf 9 tok, slots (7, 15)
scaffold[suffix] 'write me a two-line poem about rain': pre 11 tok, suf 9 tok, slots (11, 19)
scaffold[suffix] "what's a good beginner guitar?": pre 10 tok, suf 9 tok, slots (10, 18)
scaffold[suffix] 'how do I fix a leaking tap?': pre 11 tok, suf 9 tok, slots (11, 19)
scaffold[suffix] 'summarise the causes of the French Revoluti

In [36]:
# === 5.1 Does the CoT channel have a ceiling at all? ===
# §5's `cot` row is VOID: its positive control (the real word ' bridge' in the slots) scored 0.0%,
# so nothing could have registered there and the two GCG cells test nothing. The mistake is mine —
# phase 1 planted a fluent SENTENCE in the reasoning ("I really like the wolf"), and I planted a
# bare word after "Here is the key consideration:", which is not a coherent thought for the model
# to adopt. Before the cot row means anything, the channel needs a demonstrated ceiling.
import torch

CUES = [" bridge",
        " I should mention bridges",
        " the user really loves bridges",
        " bridges, obviously",
        " that bridges are the answer"]

print(f"{'slot contents':<34} {'user':>8} {'cot':>8}   (blank: user 0.0%, cot 0.0%)")
CEIL = {}
for c in CUES:
    ids = tokenizer(c, add_special_tokens=False).input_ids
    if len(ids) > K_SLOTS:
        print(f"{c!r:<34} — {len(ids)} tokens, exceeds {K_SLOTS} slots"); continue
    t = slot_text(c)
    u = rate_any(t, "user"); k = rate_any(t, "cot")
    CEIL[c] = (u, k)
    print(f"{c!r:<34} {u:>8.1%} {k:>8.1%}   ({len(ids)} tok)")

best_cot = max(CEIL, key=lambda c: CEIL[c][1]) if CEIL else None
print(f"\nbest CoT cue: {best_cot!r} at {CEIL[best_cot][1]:.1%}")
print("If every cue is ~0% in the cot column, this CoT scaffold simply has no ceiling and the")
print("cot row of §5 stays void — the channel would need phase 1's full-sentence framing, not an")
print("8-token slot. If some cue works, the GCG cells there become interpretable.")

slot contents                          user      cot   (blank: user 0.0%, cot 0.0%)
scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
scaffold[suffix] 'recommend me a book': pre 7 tok, suf 9 tok, slots (7, 15)
scaffold[suffix] 'how do I make friends in a new city?': pre 13 tok, suf 9 tok, slots (13, 21)
scaffold[suffix] 'what should I get my brother for his birthday?': pre 13 tok, suf 9 tok, slots (13, 21)
scaffold[suffix] 'explain photosynthesis briefly': pre 7 tok, suf 9 tok, slots (7, 15)
scaffold[suffix] 'write me a two-line poem about rain': pre 11 tok, suf 9 tok, slots (11, 19)
scaffold[suffix] "what's a good beginner guitar?": pre 10 tok, suf 9 tok, slots (10, 18)
scaffold[suffix] 'how do I fix a leaking tap?': pre 11 tok, suf 9 tok, slots (11, 19)
scaffold[suffix] 'summarise the causes of the French Revolution': pre 12 tok, suf 9 tok, slots (12, 20)
scaffold[suffix] "what's the difference between TCP and UDP?": pre 12 tok, suf 9 tok, slots (12, 20)
s

In [38]:
# === 6. Fluency-regularised, multi-prompt GCG ===
# Two fixes the adversarial-prompt literature already has, aimed at §5.1's finding that a FLUENT
# phrase in these slots reaches 100% while optimised gibberish reaches 0%.
#
#  (a) MULTI-PROMPT. Zou et al. optimise one suffix jointly over many prompts; I optimised on a
#      single reference prompt and evaluated on twelve. That is a methodological error, not a
#      missing extension. Objective is now averaged over 4 TRAINING prompts, disjoint from HELD_OUT.
#  (b) FLUENCY. AutoDAN / COLD-Attack / FLRT / AdvPrompter all add a perplexity term to the cost —
#      their motive is evading perplexity filters, ours is that fluent text is where the working
#      interventions live. One extra term, computed from the same forward pass:
#
#      score = mean_i [ log p(target continuation | prompt_i, trigger) ]
#            + lam * mean_i [ log p(trigger tokens | prompt_i) ]
#
# Pool is the WORD pool throughout — fluency over weakest-norm junk is not reachable.
import torch, torch.nn.functional as F

TRAIN = ["what's a fun thing to do this weekend?", "how do I get better at cooking?",
         "tell me something interesting", "what's a good habit to start?"]
assert not (set(TRAIN) & set(HELD_OUT)), "training prompts must be disjoint from evaluation"
TRAIN_SCAF = []
for p in TRAIN:
    a, b = parts("suffix", p)
    TRAIN_SCAF.append((_ids1(a), _ids1(b)))
print(f"{len(TRAIN)} training prompts, {len(HELD_OUT)} held-out eval prompts, disjoint\n")

def make_score_fn_multi(lam, chunk=16):
    @torch.no_grad()
    def f(trigs):
        B = trigs.shape[0]; tot = torch.zeros(B)
        for PREi, SUFi in TRAIN_SCAF:
            nP = PREi.shape[1]
            for i in range(0, B, chunk):
                blk = trigs[i:i+chunk].to(model.device); b = blk.shape[0]
                ids = torch.cat([PREi.expand(b, -1), blk, SUFi.expand(b, -1),
                                 TGT_IDS[None].expand(b, -1)], dim=1)
                lg = model(input_ids=ids, use_cache=False).logits
                lt = torch.log_softmax(lg[:, -NT-1:-1].float(), -1) \
                        .gather(2, TGT_IDS[None, :, None].expand(b, -1, -1)).squeeze(-1).mean(-1)
                lf = torch.log_softmax(lg[:, nP-1:nP+K_SLOTS-1].float(), -1) \
                        .gather(2, blk[:, :, None]).squeeze(-1).mean(-1)
                tot[i:i+b] += (lt + lam * lf).cpu()
                del ids, lg, lt, lf, blk
        return tot / len(TRAIN_SCAF)
    return f

def make_grad_fn_multi(lam):
    def f(trig):
        acc = None
        for PREi, SUFi in TRAIN_SCAF:
            nP = PREi.shape[1]
            oh  = F.one_hot(trig.to(model.device), V).to(E.dtype).requires_grad_(True)
            emb = torch.cat([E[PREi[0]], oh @ E, E[SUFi[0]], E[TGT_IDS]], dim=0)[None]
            lg  = model(inputs_embeds=emb, use_cache=False).logits[0]
            lt  = torch.log_softmax(lg[-NT-1:-1].float(), -1).gather(1, TGT_IDS[:, None]).mean()
            lf  = (torch.log_softmax(lg[nP-1:nP+K_SLOTS-1].float(), -1) * oh.float()).sum(-1).mean()
            (g,) = torch.autograd.grad(-(lt + lam * lf), oh)
            acc = g.detach().float().cpu() if acc is None else acc + g.detach().float().cpu()
            del oh, emb, lg, lt, lf
        return acc / len(TRAIN_SCAF)
    return f

@torch.no_grad()
def trigger_ppl(trig):
    """Perplexity of the trigger tokens in context — the thing lam is buying."""
    a, _ = parts("suffix", REF_PROMPT); P = _ids1(a); nP = P.shape[1]
    ids = torch.cat([P, trig[None].to(model.device)], dim=1)
    lg = model(input_ids=ids, use_cache=False).logits
    lp = torch.log_softmax(lg[0, nP-1:nP+K_SLOTS-1].float(), -1).gather(1, trig[:, None].to(model.device))
    return float(torch.exp(-lp.mean()))

use_word_pool()
print(f"\nreference points: fluent phrase ' the user really loves bridges' 100.0% | "
      f"' bridge' 48.6% | §3.6 single-prompt gibberish 0.0%\n")

FLU = {}
for lam in (0.0, 0.3, 1.0, 3.0):
    print(f"{'='*78}\nlam = {lam}")
    r = search(make_score_fn_multi(lam), make_grad_fn_multi(lam), k=K_SLOTS, steps=50,
               batch=64, seed=1, log_every=25, clean_fn=trigger_is_clean)
    rate = rate_any(r["trigger"], "user")
    FLU[lam] = dict(res=r, rate=rate, ppl=trigger_ppl(r["trigger"]))
    print(f"  trigger {tokenizer.decode(r['trigger'].tolist())!r}")
    print(f"  score {r['score']:.3f}  trigger-ppl {FLU[lam]['ppl']:.1f}  ON-TOPIC {rate:.1%}")

print("\n" + "=" * 78)
print(f"{'lam':>6} {'score':>9} {'trigger ppl':>13} {'on-topic':>10}   trigger")
for lam, d in FLU.items():
    print(f"{lam:>6} {d['res']['score']:>9.3f} {d['ppl']:>13.1f} {d['rate']:>10.1%}   "
          f"{tokenizer.decode(d['res']['trigger'].tolist())!r}")

4 training prompts, 12 held-out eval prompts, disjoint

  word pool: 63971 tokens  e.g. [' suit', ' smart', 'oma', ' Exp', ' assist', ' successfully', 'sys', ' Cre', ' Thursday', ' bur']

reference points: fluent phrase ' the user really loves bridges' 100.0% | ' bridge' 48.6% | §3.6 single-prompt gibberish 0.0%

lam = 0.0
  step   0  score=-7.8077  ' please embracepared attacking Dareecha obr secondo'
  step  25  score=-5.5868  ' pleaselavadinlemma tek muj treffredni'
  step  49  score=-5.5288  ' please SVMatiflemma tek muj treffredni'
scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
scaffold[suffix] 'recommend me a book': pre 7 tok, suf 9 tok, slots (7, 15)
scaffold[suffix] 'how do I make friends in a new city?': pre 13 tok, suf 9 tok, slots (13, 21)
scaffold[suffix] 'what should I get my brother for his birthday?': pre 13 tok, suf 9 tok, slots (13, 21)
scaffold[suffix] 'explain photosynthesis briefly': pre 7 tok, suf 9 tok, slots (7, 15)
scaffold[suffix]

In [34]:
try:
    for k,v in CTRL.items(): print("CTRL", k, {a: f"{b:.1%}" for a,b in v.items()})
except NameError: print("controls still running")
try:
    for k,v in GRID.items(): print("GRID", k, f"score {v['res']['score']:.3f}  rate {v['rate']:.1%}")
except NameError: print("grid not started")

CTRL user {'blank': '0.0%', 'real': '48.6%'}
CTRL cot {'blank': '0.0%', 'real': '0.0%'}
GRID ('user', 'junk') score -3.875  rate 0.0%
GRID ('user', 'words') score -5.479  rate 0.0%
GRID ('cot', 'junk') score -4.232  rate 0.0%
GRID ('cot', 'words') score -5.043  rate 0.0%


In [43]:
# === 7. Match quality across the FULL depth, not just the optimised layer ===
# §3.4/§3.5 optimised the match at ONE layer (35). The obvious explanation for 0% behaviour was a
# keyhole: GCG matches at 35 and diverges everywhere else, so the "match" is an artifact of where
# we looked. THAT EXPLANATION IS WRONG — measured below, and worth recording because it was the
# natural hypothesis.
#
# Profile cos(delta_candidate, delta*) at every layer downstream of the injection, for five
# interventions whose free-generation behaviour we already know.
import torch, math

SCORE_MODE = "post"
set_scaffold("suffix", REF_PROMPT)

CANDS = {                                                  # name -> (slot contents, known rate)
    "real word ' bridge'":  (slot_text(" bridge"),                            0.486),
    "fluent phrase":        (slot_text(" the user really loves bridges"),     1.000),
    "cos-opt trigger":      (res_cos["trigger"],                              0.000),
    "proj-opt trigger":     (res["trigger"],                                  0.000),
    "random pool trigger":  (rand[0],                                         0.000),
}
LAYERS_PROFILE = list(range(L_TARGET + 1, N_LAYERS, 2))

print(f"cos(delta_candidate, delta*) by layer — objective was optimised at L{LAYER_BEST} only\n")
print(f"{'layer':>6} " + " ".join(f"{n[:15]:>16}" for n in CANDS))
PROF = {n: [] for n in CANDS}
for L in LAYERS_PROFILE:
    ref_L = make_reference(REF_PROMPT, "suffix", L, where="span", verbose=False)
    row = []
    for n, (t, _) in CANDS.items():
        c = report(t, ref_L)["cos"]; PROF[n].append(c); row.append(f"{c:>+16.3f}")
    print(f"{L:>6} " + " ".join(row))

print(f"\n{'candidate':<22} {'mean':>8} {'L17-31':>8} {'L35':>8} {'behaviour':>10}")
for n, (_, rate) in CANDS.items():
    v = PROF[n]
    print(f"{n:<22} {sum(v)/len(v):>+8.3f} {sum(v[:-2])/len(v[:-2]):>+8.3f} {v[-1]:>+8.3f} {rate:>10.1%}")

def pear(a, b):
    n = len(a); ma, mb = sum(a)/n, sum(b)/n
    sa = math.sqrt(sum((x-ma)**2 for x in a)); sb = math.sqrt(sum((y-mb)**2 for y in b))
    return sum((x-ma)*(y-mb) for x, y in zip(a, b)) / (sa*sb) if sa and sb else float("nan")

mc = [sum(PROF[n])/len(PROF[n]) for n in CANDS]
bh = [CANDS[n][1] for n in CANDS]
print(f"\ncorr(mean match across depth, behaviour) = {pear(mc, bh):+.3f}   (n=5 — indicative only)")

print("""
What this rules out, and what it does NOT show:

  NOT a keyhole. The GCG triggers track the steering vector across the WHOLE depth profile, with
  the same shape as the real word's — they are not matching at L35 and diverging elsewhere. That
  was the natural hypothesis and it is wrong.

  The intervention that reaches 100% matches WORST of the non-random candidates at every depth,
  sitting near the random floor through the mid layers. Better match does not mean better
  behaviour at ANY depth, not merely at the optimised one.

  BUT the correlation here is ~0, i.e. UNCORRELATED, not anti-correlated — the real word has both
  a high match and a decent rate, which offsets the fluent phrase. The anti-correlation (-0.55,
  -0.60) belongs to §6's teacher-forced NLL proxy, which is a different measure. Two proxies fail
  in two different ways: activation match is uninformative, target NLL is actively inverted.

  Reading: the steering vector's activation signature is not what produces the behaviour. A fluent
  statement of preference and an activation edit are two different internal routes to the same
  output, so matching the vector makes a trigger vector-like rather than bridge-inducing.
  n=5 interventions — indicative, not a measured correlation.""")

scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
cos(delta_candidate, delta*) by layer — objective was optimised at L35 only

 layer  real word ' bri    fluent phrase  cos-opt trigger  proj-opt trigge  random pool tri
scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
    17           +0.095           +0.049           +0.072           +0.069           +0.087
scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
    19           +0.074           -0.005           +0.069           +0.028           +0.078
scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
    21           +0.182           +0.126           +0.189           +0.198           +0.165
scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8, 16)
    23           +0.224           +0.154           +0.231           +0.210           +0.210
scaffold[suffix] 'what shall i do today': pre 8 tok, suf 9 tok, slots (8,